# 1. Carga de datos procesados y sucursales

Este notebook parte de los datasets ya limpios y con `precio_efectivo` calculado en la etapa anterior (ver `01_analisis_dia_carrefour.ipynb`).  
En esta fase se trabaja sobre:

- los **productos comunes** entre DIA y Carrefour, ya procesados y guardados en formato Parquet;
- los **datasets de sucursales** de cada cadena, que permiten incorporar información geográfica y organizacional (bandera, provincia, localidad, etc.).

In [278]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [279]:
# Cargamos los datos procesados de productos de DIA y productos de Carrefour
productos_dia_comunes = pd.read_parquet('../data/processed/productos_dia_procesado.parquet')
productos_carrefour_comunes = pd.read_parquet('../data/processed/productos_carrefour_procesado.parquet')

In [280]:
# Cargamos los datasets como texto para conservar el formato original de
# identificadores y evitar conversiones automáticas de tipos.
sucursales_dia = pd.read_csv(
    '../data/raw/sucursales_dia.csv',
    delimiter='|',
    encoding='UTF-8',
    dtype=str,
    low_memory=False
)

sucursales_carrefour = pd.read_csv(
    '../data/raw/sucursales_carrefour.csv',
    delimiter='|',
    encoding='UTF-8',
    dtype=str,
    low_memory=False
)

In [281]:
# Comprobamos las dimensiones iniciales de ambos datasets.
print("Dimensiones de sucursales_dia:", sucursales_dia.shape)
print("Dimensiones de sucursales_carrefour:", sucursales_carrefour.shape)

Dimensiones de sucursales_dia: (984, 21)
Dimensiones de sucursales_carrefour: (731, 21)


Tras la carga inicial se inspeccionan las primeras y últimas filas de cada dataset, lo que permite:

- verificar la estructura de las columnas;
- confirmar la presencia de la fila de metadatos al final del archivo (similar a la observada en los datasets de productos);

In [282]:
# Mostramos las primeras y últimas filas de ambos datasets para inspección visual.
pd.set_option('display.max_columns', None)  # Mostrar todas las columnas

print("Primeras 5 filas de sucursales_dia:")
display(sucursales_dia.head(5))

print("Últimas 5 filas de sucursales_dia:")
display(sucursales_dia.tail(5))

print("Primeras 10 filas de sucursales_carrefour:")
display(sucursales_carrefour.head(5))

print("Últimas 10 filas de sucursales_carrefour:")
display(sucursales_carrefour.tail(5))

Primeras 5 filas de sucursales_dia:


,id_comercio,id_bandera,id_sucursal,sucursales_nombre,sucursales_tipo,sucursales_calle,sucursales_numero,sucursales_latitud,sucursales_longitud,sucursales_observaciones,sucursales_barrio,sucursales_codigo_postal,sucursales_localidad,sucursales_provincia,sucursales_lunes_horario_atencion,sucursales_martes_horario_atencion,sucursales_miercoles_horario_atencion,sucursales_jueves_horario_atencion,sucursales_viernes_horario_atencion,sucursales_sabado_horario_atencion,sucursales_domingo_horario_atencion
0,15,1,1,1 - VILLA DEL PARQUE,Autoservicio,CL BEIRO,3146,-34.5971373,-58.4976429,NaN,VILLA DEL PARQUE,00001419,CAPITAL FEDERAL,AR-C,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00
1,15,1,3,3 - BALVANERA,Autoservicio,CL ENTRE RIOS,735,-34.6173202,-58.3916913,NaN,BALVANERA,00001080,CAPITAL FEDERAL,AR-C,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00
2,15,1,4,4 - PARQUE PATRICIOS,Autoservicio,CL CASEROS,2938,-34.6369605,-58.4045551,NaN,PARQUE PATRICIOS,00001264,CAPITAL FEDERAL,AR-C,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00
3,15,1,5,5 - SAN CRISTÓBAL,Autoservicio,AV SAN JUAN,2641,-34.6236734,-58.4014697,NaN,SAN CRISTÓBAL,00001232,CAPITAL FEDERAL,AR-C,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00
4,15,1,6,6 - FLORES,Autoservicio,CL ARTIGAS,26,-34.6289482,-58.4642662,NaN,FLORES,00001406,CAPITAL FEDERAL,AR-C,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00


Últimas 5 filas de sucursales_dia:


,id_comercio,id_bandera,id_sucursal,sucursales_nombre,sucursales_tipo,sucursales_calle,sucursales_numero,sucursales_latitud,sucursales_longitud,sucursales_observaciones,sucursales_barrio,sucursales_codigo_postal,sucursales_localidad,sucursales_provincia,sucursales_lunes_horario_atencion,sucursales_martes_horario_atencion,sucursales_miercoles_horario_atencion,sucursales_jueves_horario_atencion,sucursales_viernes_horario_atencion,sucursales_sabado_horario_atencion,sucursales_domingo_horario_atencion
979,15,1,8014,8014 - METAN,Autoservicio,JOSE IGNACIO SIERRA,126,-25.4987985,-64.9705319,NaN,METAN,00004440,SALTA,AR-A,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00
980,15,1,8017,8017 - GRAL. GUEMES,Autoservicio,CABRED,23,-24.671542,-65.045994,NaN,GRAL. GUEMES,00004430,SALTA,AR-A,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00
981,15,1,8018,8018 - JUJUY,Autoservicio,BELGRANO,620,-24.387603,-65.264866,NaN,JUJUY,00004603,JUJUY,AR-J,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00
982,15,1,90000,eCommerce,Supermercado,Juan Francisco Segui,4646,-34.5736049,-58.4232349,NaN,NaN,00001425,CABA,Buenos Aires,00:00 a 24:00,00:00 a 24:00,00:00 a 24:00,00:00 a 24:00,00:00 a 24:00,00:00 a 24:00,00:00 a 24:00
983,Última actualización: 2026-07-03T04:05:37-03:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Primeras 10 filas de sucursales_carrefour:


,id_comercio,id_bandera,id_sucursal,sucursales_nombre,sucursales_tipo,sucursales_calle,sucursales_numero,sucursales_latitud,sucursales_longitud,sucursales_observaciones,sucursales_barrio,sucursales_codigo_postal,sucursales_localidad,sucursales_provincia,sucursales_lunes_horario_atencion,sucursales_martes_horario_atencion,sucursales_miercoles_horario_atencion,sucursales_jueves_horario_atencion,sucursales_viernes_horario_atencion,sucursales_sabado_horario_atencion,sucursales_domingo_horario_atencion
0,10,3,365,Nazca 1944,Autoservicio,Nazca,1944,-34.612145,-58.480997,NaN,Santa Rita,1416,Ciudad Autónoma de Buenos Aires,AR-C,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,10:30 a 20:00
1,10,3,408,Perú 916,Autoservicio,Perú,916,-34.618745,-58.374322,NaN,San telmo,1068,Ciudad Autónoma de Buenos Aires,AR-C,07:00 a 20:00,07:00 a 20:00,07:00 a 20:00,07:00 a 20:00,07:00 a 20:00,07:00 a 20:00,11:00 a 20:00
2,10,4,504,Maxi Cacheuta Cba (504),Hipermercado,Cacheuta 4419,4419,-31.460585,-64.215363,NaN,NaN,5016,Cacheuta,AR-C,08:00 A 22:00,08:00 A 22:00,08:00 A 22:00,08:00 A 22:00,08:00 A 22:00,08:00 A 22:00,08:00 A 22:00
3,10,3,755,C.Tribulato 1279 (San Miguel),Autoservicio,C Tribulato,1279,-34.541507,-58.713798,NaN,NaN,1663,San Miguel,AR-B,07:00 A 22:00,07:00 A 22:00,07:00 A 22:00,07:00 A 22:00,07:00 A 22:00,07:00 A 22:00,09:00 A 20:00
4,10,3,479,Avenida Eva Perón 4215,Autoservicio,Avenida Eva Perón,4215,-34.690047,-58.689142,NaN,NaN,1722,Merlo,AR-B,07:30 a 20:00,07:30 a 20:00,07:30 a 20:00,07:30 a 20:00,07:30 a 20:00,07:30 a 20:00,10:00 a 19:00


Últimas 10 filas de sucursales_carrefour:


,id_comercio,id_bandera,id_sucursal,sucursales_nombre,sucursales_tipo,sucursales_calle,sucursales_numero,sucursales_latitud,sucursales_longitud,sucursales_observaciones,sucursales_barrio,sucursales_codigo_postal,sucursales_localidad,sucursales_provincia,sucursales_lunes_horario_atencion,sucursales_martes_horario_atencion,sucursales_miercoles_horario_atencion,sucursales_jueves_horario_atencion,sucursales_viernes_horario_atencion,sucursales_sabado_horario_atencion,sucursales_domingo_horario_atencion
726,10,3,344,M. T. de Alvear 2251,Autoservicio,Marcelo T. de Alvear,2251,-34.596862,-58.399571,NaN,Recoleta,1122,Ciudad Autónoma de Buenos Aires,AR-C,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,11:00 a 20:00
727,10,3,593,Entre Ríos 1192,Autoservicio,Entre Ríos,1192,-34.622603,-58.391341,NaN,San Cristobal,1080,Ciudad Autónoma de Buenos Aires,AR-C,07:30 a 20:00,07:30 a 20:00,07:30 a 20:00,07:30 a 20:00,07:30 a 20:00,07:30 a 20:00,11:00 a 20:00
728,10,3,791,BEIRO 3342,Autoservicio,BEIRO,3345,-34.598084,-58.500119,NaN,NaN,1419,Ciudad Autónoma de Buenos Aires,AR-C,07:00 A 22:00,07:00 A 22:00,07:00 A 22:00,07:00 A 22:00,07:00 A 22:00,07:00 A 22:00,11:00 A 20:00
729,10,3,608,Serrano 280,Autoservicio,Serrano,280,-34.598185,-58.444983,NaN,Villa Crespo,1414,Ciudad Autónoma de Buenos Aires,AR-C,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,11:00 a 20:00
730,Ultima actualización: 2026-07-03T08:30:05-03:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Limpieza inicial y control de calidad

La inspección inicial de los datasets de sucursales muestra que la columna `sucursales_provincia` utiliza códigos ISO 3166‑2, el estándar internacional para identificar subdivisiones dentro de un país. En este formato, la parte posterior al guion corresponde a la provincia dentro de Argentina, por lo que esta columna resulta fundamental para uno de los objetivos centrales del análisis: comparar precios por provincia.

Durante la revisión también se detectó una inconsistencia en una de las filas, donde aparece el nombre “Buenos Aires” en lugar del código ISO correspondiente. Este tipo de irregularidades requiere normalización para asegurar que todas las provincias estén correctamente codificadas y que los análisis posteriores —especialmente los que involucran comparaciones regionales— se basen en información homogénea y confiable.

Además, se observa una fila final de metadatos similar a la presente en los datasets de productos. Esta fila no representa una sucursal real y se elimina antes de continuar, de modo que el análisis posterior se realice únicamente sobre registros válidos.

In [283]:
# Eliminamos filas con valores nulos en la columna 'id_sucursal' para ambos datasets
sucursales_dia = sucursales_dia.dropna(subset=['id_sucursal']).copy()
sucursales_carrefour = sucursales_carrefour.dropna(subset=['id_sucursal']).copy()

### 2.1 Unicidad de la columna `id_sucursal` en ambos datasets.

In [284]:
# Comprobación de unicidad de la columna 'id_sucursal' en ambos datasets
print("DIA — id_sucursal único:", sucursales_dia['id_sucursal'].is_unique)
print("Carrefour — id_sucursal único:", sucursales_carrefour['id_sucursal'].is_unique)

DIA — id_sucursal único: True
Carrefour — id_sucursal único: True


El identificador id_sucursal es único dentro de cada cadena. Por lo tanto, puede utilizarse como clave para integrar la información de sucursales con los registros de productos correspondientes a la misma cadena.

### 2.3 Valores nulos

In [285]:
# Comprobación de valores nulos en ambos datasets
print("Nulos en sucursales DIA:")
print(sucursales_dia.isna().sum()[lambda s: s > 0])

print("\nNulos en sucursales Carrefour:")
print(sucursales_carrefour.isna().sum()[lambda s: s > 0])

Nulos en sucursales DIA:
sucursales_numero           429
sucursales_observaciones    983
sucursales_barrio             1
dtype: int64

Nulos en sucursales Carrefour:
sucursales_observaciones                 713
sucursales_barrio                        428
sucursales_codigo_postal                  10
sucursales_provincia                      36
sucursales_lunes_horario_atencion         36
sucursales_martes_horario_atencion        36
sucursales_miercoles_horario_atencion     36
sucursales_jueves_horario_atencion        36
sucursales_viernes_horario_atencion       36
sucursales_sabado_horario_atencion        36
sucursales_domingo_horario_atencion       36
dtype: int64


En **DIA** los nulos se concentran en campos no críticos para el análisis geográfico (`sucursales_numero`, `sucursales_observaciones`, `sucursales_barrio`).

En **Carrefour** aparecen **36 nulos en `sucursales_provincia`**, y esa misma cantidad se repite en las columnas de horarios de atención. El patrón indica **filas incompletas**, no valores faltantes aislados: son registros cargados parcialmente. Como la provincia es la variable central de esta etapa, se recuperan en la sección 3.

## 3. Imputación de la provincia faltante en Carrefour

Los 36 registros se resuelven con una **estrategia en cascada**: cada método se aplica sólo a los casos que el anterior no resolvió, ordenados de la información más directa a la más inferida.

1. **Coincidencia exacta con el nombre de una provincia** en `sucursales_localidad`.
2. **Diccionario de ciudades conocidas**, cuando la localidad identifica la provincia sin ambigüedad.
3. **Inferencia por coordenadas**, para los registros sin localidad informada.

In [286]:
# Convertimos las columnas de latitud y longitud a tipo numérico, 
# manejando errores con 'coerce' para convertir valores no numéricos a NaN.
sucursales_carrefour['sucursales_latitud'] = pd.to_numeric(
    sucursales_carrefour['sucursales_latitud'], errors='coerce'
)

sucursales_carrefour['sucursales_longitud'] = pd.to_numeric(
    sucursales_carrefour['sucursales_longitud'], errors='coerce'
)

In [287]:
carrefour_prov_nulos = sucursales_carrefour[sucursales_carrefour['sucursales_provincia'].isna()].copy()
carrefour_prov_nulos


,id_comercio,id_bandera,id_sucursal,sucursales_nombre,sucursales_tipo,sucursales_calle,sucursales_numero,sucursales_latitud,sucursales_longitud,sucursales_observaciones,sucursales_barrio,sucursales_codigo_postal,sucursales_localidad,sucursales_provincia,sucursales_lunes_horario_atencion,sucursales_martes_horario_atencion,sucursales_miercoles_horario_atencion,sucursales_jueves_horario_atencion,sucursales_viernes_horario_atencion,sucursales_sabado_horario_atencion,sucursales_domingo_horario_atencion
32,10,3,6018,"Viamonte 5366, Luján de Cuyo, Mendoza",Autoservicio,Viamonte 5366,5366,-32.986263,-68.882983,NaN,NaN,5505,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
62,10,3,6009,"Balcarce 401, Ciudad de Salta",Autoservicio,Balcarce 401,401,-24.784593,-65.412082,NaN,NaN,4400,SALTA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
72,10,3,6016,"Boedo 2879, Luján de Cuyo, Mendoza",Autoservicio,Boedo 2879,2879,-32.987800,-68.829942,NaN,NaN,5503,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
132,10,3,6023,"Av San Martin 792, Ciudad de Mendoza",Autoservicio,Av San Martin 792,792,-32.893272,-68.840495,NaN,NaN,5500,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
133,10,3,6025,"Aristides 287, Ciudad de Mendoza",Autoservicio,Aristides 287,287,-32.891987,-68.854527,NaN,NaN,5500,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
138,10,3,6022,"Hipolito Yrigoyen 2695, Maipú, Mendoza (Comple...",Autoservicio,Hipolito Yirigoyen 2695,2695,-32.959835,-68.783645,NaN,NaN,5515,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
151,10,3,6005,MINI - RIVADEO 1487,Autoservicio,Rivadeo 1487,1487,-31.400816,-64.181451,NaN,NaN,NaN,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
168,10,3,6032,"Avenida Boyacá 613, Capital Federal",Autoservicio,Avenida Boyacá 613,613,-34.620054,-58.461094,NaN,NaN,1406,Ciudad Autónoma de Buenos Aires,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
181,10,3,6007,MINI - J B ALBERDI 3099,Autoservicio,Juan Bautista Alberdi 3099,3099,-34.633684,-58.471208,NaN,NaN,NaN,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
225,10,3,6012,"Avenida Bordereau 9950, Villa Allende, Córdoba...",Autoservicio,Avenida Bordereau 9950,9950,-31.309289,-64.287994,NaN,NaN,5105,CORDOBA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


La revisión de los valores faltantes muestra que la mayoría de los registros con `sucursales_provincia` nulo pueden recuperarse utilizando la información disponible en `sucursales_localidad`, donde en muchos casos aparece directamente el nombre de la provincia. En otros registros, la localidad corresponde a una ciudad, lo que permite deducir la provincia de forma confiable. Para los pocos casos restantes, la provincia puede inferirse a partir del nombre de la sucursal o, en última instancia, mediante la ubicación geográfica utilizando `sucursales_latitud` y `sucursales_longitud`. De este modo, es posible completar los 36 registros incompletos antes de continuar con el análisis regional.

In [288]:
# Vemos las columnas relevantes de las sucursales de Carrefour que tienen valores nulos en la columna 'sucursales_provincia'.
carrefour_prov_nulos[['id_sucursal', 'sucursales_nombre', 'sucursales_localidad',
                      'sucursales_latitud', 'sucursales_longitud']]

,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_latitud,sucursales_longitud
32,6018,"Viamonte 5366, Luján de Cuyo, Mendoza",Mendoza,-32.986263,-68.882983
62,6009,"Balcarce 401, Ciudad de Salta",SALTA,-24.784593,-65.412082
72,6016,"Boedo 2879, Luján de Cuyo, Mendoza",Mendoza,-32.987800,-68.829942
132,6023,"Av San Martin 792, Ciudad de Mendoza",Mendoza,-32.893272,-68.840495
133,6025,"Aristides 287, Ciudad de Mendoza",Mendoza,-32.891987,-68.854527
138,6022,"Hipolito Yrigoyen 2695, Maipú, Mendoza (Comple...",Mendoza,-32.959835,-68.783645
151,6005,MINI - RIVADEO 1487,-,-31.400816,-64.181451
168,6032,"Avenida Boyacá 613, Capital Federal",Ciudad Autónoma de Buenos Aires,-34.620054,-58.461094
181,6007,MINI - J B ALBERDI 3099,-,-34.633684,-58.471208
225,6012,"Avenida Bordereau 9950, Villa Allende, Córdoba...",CORDOBA,-31.309289,-64.287994


### 3.1 Nivel 1 — La localidad coincide con el nombre de una provincia

In [289]:
# Identificamos las sucursales de Carrefour donde la localidad coincide con el nombre de la provincia.
# Para ello, creamos una lista de provincias argentinas (sin tildes)
# y comparamos con la columna 'sucursales_localidad' del dataset de Carrefour.
provincias_arg = [
    'Buenos Aires', 'CABA', 'Catamarca', 'Chaco', 'Chubut', 'Cordoba',
    'Corrientes', 'Entre Rios', 'Formosa', 'Jujuy', 'La Pampa', 'La Rioja',
    'Mendoza', 'Misiones', 'Neuquen', 'Rio Negro', 'Salta', 'San Juan',
    'San Luis', 'Santa Cruz', 'Santa Fe', 'Santiago del Estero',
    'Tierra del Fuego', 'Tucuman'
]

# Normalizamos la columna 'sucursales_localidad' 
carrefour_prov_nulos['sucursales_localidad_norm'] = (
    carrefour_prov_nulos['sucursales_localidad']
    .str.upper()
    .str.strip()
)
# Normalizamos la lista de provincias para compararlas en mayúsculas
provincias_arg_upper = [p.upper() for p in provincias_arg]

carrefour_localidad_es_provincia = carrefour_prov_nulos[
    carrefour_prov_nulos['sucursales_localidad_norm'].isin(provincias_arg_upper)
]

print(f'Número de sucursales donde la ciudad comparte el nombre de su provincia: {carrefour_localidad_es_provincia["sucursales_localidad"].count()}')

carrefour_localidad_es_provincia


Número de sucursales donde la ciudad comparte el nombre de su provincia: 25


,id_comercio,id_bandera,id_sucursal,sucursales_nombre,sucursales_tipo,sucursales_calle,sucursales_numero,sucursales_latitud,sucursales_longitud,sucursales_observaciones,sucursales_barrio,sucursales_codigo_postal,sucursales_localidad,sucursales_provincia,sucursales_lunes_horario_atencion,sucursales_martes_horario_atencion,sucursales_miercoles_horario_atencion,sucursales_jueves_horario_atencion,sucursales_viernes_horario_atencion,sucursales_sabado_horario_atencion,sucursales_domingo_horario_atencion,sucursales_localidad_norm
32,10,3,6018,"Viamonte 5366, Luján de Cuyo, Mendoza",Autoservicio,Viamonte 5366,5366,-32.986263,-68.882983,NaN,NaN,5505,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MENDOZA
62,10,3,6009,"Balcarce 401, Ciudad de Salta",Autoservicio,Balcarce 401,401,-24.784593,-65.412082,NaN,NaN,4400,SALTA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SALTA
72,10,3,6016,"Boedo 2879, Luján de Cuyo, Mendoza",Autoservicio,Boedo 2879,2879,-32.987800,-68.829942,NaN,NaN,5503,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MENDOZA
132,10,3,6023,"Av San Martin 792, Ciudad de Mendoza",Autoservicio,Av San Martin 792,792,-32.893272,-68.840495,NaN,NaN,5500,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MENDOZA
133,10,3,6025,"Aristides 287, Ciudad de Mendoza",Autoservicio,Aristides 287,287,-32.891987,-68.854527,NaN,NaN,5500,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MENDOZA
138,10,3,6022,"Hipolito Yrigoyen 2695, Maipú, Mendoza (Comple...",Autoservicio,Hipolito Yirigoyen 2695,2695,-32.959835,-68.783645,NaN,NaN,5515,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MENDOZA
225,10,3,6012,"Avenida Bordereau 9950, Villa Allende, Córdoba...",Autoservicio,Avenida Bordereau 9950,9950,-31.309289,-64.287994,NaN,NaN,5105,CORDOBA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CORDOBA
243,10,3,6030,"Avenida Castro Barros 858, Córdoba Capital",Autoservicio,Avenida Castro Barros 858,858,-31.397186,-64.199161,NaN,NaN,5000,CORDOBA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CORDOBA
255,10,3,6031,"Italia 5833, Luján de Cuyo, Mendoza",Autoservicio,Italia 5833,5833,-32.982480,-68.878215,NaN,NaN,5505,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MENDOZA
290,10,3,6017,"Viamonte 4620, Luján de Cuyo, Mendoza",Autoservicio,Viamonte 4620,4620,-32.992785,-68.887172,NaN,NaN,5505,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MENDOZA


Del análisis de los 36 registros con `sucursales_provincia` nulo se observa que, en 25 de ellos, la columna `sucursales_localidad` contiene directamente el nombre de la provincia. Tras normalizar el texto (mayúsculas y eliminación de espacios), estas localidades coinciden de manera exacta con los nombres de las provincias argentinas, por lo que estos casos pueden imputarse de forma segura y sin ambigüedad.

Completar estos 25 registros es un paso necesario antes de continuar, ya que en el resto del dataset las provincias aparecen codificadas en formato ISO 3166‑2. Una vez imputados estos valores, será posible reemplazar los códigos ISO por los nombres de las provincias de manera consistente en todo el dataset, asegurando una estructura homogénea para el análisis regional.

In [290]:
# Imputamos la provincia para las 25 sucursales donde la localidad coincide
# con el nombre de la provincia.
sucursales_carrefour.loc[
    carrefour_localidad_es_provincia.index,
    'sucursales_provincia'
] = carrefour_localidad_es_provincia['sucursales_localidad_norm']


### 3.2 Nivel 2 — La localidad es una ciudad identificable

In [291]:
# Identificamos las sucursales cuya localidad NO coincide con ninguna provincia argentina
carrefour_localidad_es_ciudad = carrefour_prov_nulos[
    ~carrefour_prov_nulos['sucursales_localidad_norm'].isin(provincias_arg_upper)
]

carrefour_localidad_es_ciudad[['sucursales_localidad', 'sucursales_nombre',
                      'sucursales_latitud', 'sucursales_longitud']]


,sucursales_localidad,sucursales_nombre,sucursales_latitud,sucursales_longitud
151,-,MINI - RIVADEO 1487,-31.400816,-64.181451
168,Ciudad Autónoma de Buenos Aires,"Avenida Boyacá 613, Capital Federal",-34.620054,-58.461094
181,-,MINI - J B ALBERDI 3099,-34.633684,-58.471208
277,-,MINI - LOS GRANADEROS 2269 CBA,-31.377924,-64.204323
330,MAR DEL PLATA,MINI - LURO 3060 (832),-37.995534,-57.550968
355,MAR DEL PLATA,ALBERTI 1511,-38.011862,-57.541844
471,MAR DEL PLATA,MINI - GUEMES 2470 (801),-38.011354,-57.538653
501,-,MINI - SAN JUAN 490 CBA,-31.418132,-64.192626
523,MAR DEL PLATA,MINI - SAN MARTIN 3579 (803),-37.994060,-57.556713
595,MAR DEL PLATA,MINI - SANTA FE 2248 (800),-38.004675,-57.548313


In [292]:
# Diccionario de ciudades para imputar provincia
ciudad_a_provincia = {
    'CIUDAD AUTÓNOMA DE BUENOS AIRES': 'CABA',
    'CIUDAD AUTONOMA DE BUENOS AIRES': 'CABA',
    'MAR DEL PLATA': 'BUENOS AIRES',
}


In [293]:
# Completamos la provincia usando el diccionario de ciudades a provincias.
sucursales_carrefour.loc[
    carrefour_localidad_es_ciudad.index,
    'sucursales_provincia'
] = carrefour_localidad_es_ciudad['sucursales_localidad_norm'].map(ciudad_a_provincia).str.upper()


### 3.3 Nivel 3 — Inferencia por coordenadas

Los registros restantes tienen la localidad informada como `'-'`. Se utilizan rangos de latitud y longitud que delimitan de forma aproximada las zonas involucradas.

> **Nota metodológica:** es una aproximación por rangos, no una georreferenciación contra límites oficiales. Es adecuada para un número acotado de casos en zonas geográficamente separadas, y no debe usarse como método general de asignación.

In [294]:
# Identificamos las sucursales de Carrefour donde la localidad es un guion ('-').
carrefour_localidad_guion = carrefour_localidad_es_ciudad[
    carrefour_localidad_es_ciudad['sucursales_localidad_norm'] == '-'
]

In [295]:
# Imputamos la provincia, según sus coordenadas geográficas,
# para las sucursales de Carrefour donde la localidad es un guion.
cordoba_ubi = (
    (carrefour_localidad_guion['sucursales_latitud'] < -30) &
    (carrefour_localidad_guion['sucursales_latitud'] > -35) &
    (carrefour_localidad_guion['sucursales_longitud'] < -61) &
    (carrefour_localidad_guion['sucursales_longitud'] > -65)
)

sucursales_carrefour.loc[
    carrefour_localidad_guion[cordoba_ubi].index,
    'sucursales_provincia'
] = 'CORDOBA'

caba_ubi = (
    (carrefour_localidad_guion['sucursales_latitud'] < -34) &
    (carrefour_localidad_guion['sucursales_latitud'] > -35) &
    (carrefour_localidad_guion['sucursales_longitud'] < -58) &
    (carrefour_localidad_guion['sucursales_longitud'] > -59)
)

sucursales_carrefour.loc[
    carrefour_localidad_guion[caba_ubi].index,
    'sucursales_provincia'
] = 'CABA'


In [296]:
# Verificamos que no queden provincias sin imputar
sucursales_carrefour['sucursales_provincia'].isna().sum()

np.int64(0)

Los 36 registros quedan completamente imputados.

Como consecuencia, `sucursales_provincia` contiene ahora **dos formatos conviviendo**: códigos ISO en los registros originales y nombres completos en los imputados. La unificación se resolverá más adelante, una vez auditados los códigos.

## 4. Carga del dataset de provincias (ISO 3166-2)

En esta sección se carga el archivo `provincias.csv`, que contiene la lista oficial de provincias argentinas junto con sus códigos ISO 3166-2.  
Este dataset se utilizará para reemplazar los códigos ISO presentes en la columna `sucursales_provincia` de los datasets de DIA y Carrefour, de modo que todas las provincias queden expresadas con sus nombres completos y normalizados.


In [297]:
# Cargamos el dataset de provincias para su posterior uso en la imputación de provincias faltantes.
provincias = pd.read_csv('../data/metadata/provincias.csv', dtype=str)

print("Dimensiones:", provincias.shape)
display(provincias)

Dimensiones: (24, 9)


,categoria,centroide_lat,centroide_lon,fuente,id,iso_id,iso_nombre,nombre,nombre_completo
0,Provincia,-26.8753965086829,-54.6516966230371,IGN,54,AR-N,Misiones,Misiones,Provincia de Misiones
1,Provincia,-33.7577257449137,-66.0281298195836,IGN,74,AR-D,San Luis,San Luis,Provincia de San Luis
2,Provincia,-30.8653679979618,-68.8894908486844,IGN,70,AR-J,San Juan,San Juan,Provincia de San Juan
3,Provincia,-32.0588735436448,-59.2014475514635,IGN,30,AR-E,Entre Ríos,Entre Ríos,Provincia de Entre Ríos
4,Provincia,-48.8154851827063,-69.9557621671973,IGN,78,AR-Z,Santa Cruz,Santa Cruz,Provincia de Santa Cruz
5,Provincia,-40.4057957178801,-67.229329893694,IGN,62,AR-R,Río Negro,Río Negro,Provincia de Río Negro
6,Provincia,-43.7886233529878,-68.5267593943345,IGN,26,AR-U,Chubut,Chubut,Provincia del Chubut
7,Provincia,-32.142932663607,-63.8017532741662,IGN,14,AR-X,Córdoba,Córdoba,Provincia de Córdoba
8,Provincia,-34.6298873058957,-68.5831228183798,IGN,50,AR-M,Mendoza,Mendoza,Provincia de Mendoza
9,Provincia,-29.685776298315,-67.1817359694432,IGN,46,AR-F,La Rioja,La Rioja,Provincia de La Rioja


> **Fuente de datos — Provincias de Argentina (ISO 3166-2)**  
> Este archivo (`provincias.csv`) proviene del Instituto Geográfico Nacional (IGN), publicado en el portal oficial de datos del Gobierno Argentino.  
> **Descarga oficial:** https://infra.datos.gob.ar/catalog/modernizacion/dataset/7/distribution/7.7/download/provincias.csv  
> Contiene la lista completa de provincias de Argentina junto con sus códigos ISO 3166-2, utilizados para normalizar y unificar referencias geográficas dentro del proyecto.


### 4.1 Exploración de los códigos ISO presentes en las sucursales

Antes de reemplazar los códigos ISO 3166-2 por los nombres de las provincias, es útil visualizar cómo aparecen estos códigos en los datasets de DIA y Carrefour.  
Para ello, se muestra un ejemplo por cada valor distinto de `sucursales_provincia`, junto con la localidad y el nombre de la sucursal.  
Esta inspección permite verificar la consistencia de los códigos y detectar posibles irregularidades antes de realizar la conversión definitiva.


In [298]:
# Inspeccionamos un ejemplo de cada código de provincia para ambos datasets.
columnas = ['sucursales_provincia', 'sucursales_localidad', 'sucursales_nombre']

print("DIA — un ejemplo por cada código:")
display(
    sucursales_dia.drop_duplicates(subset='sucursales_provincia')[columnas]
    .sort_values('sucursales_provincia')
)

print("\nCarrefour — un ejemplo por cada código:")
display(
    sucursales_carrefour.drop_duplicates(subset='sucursales_provincia')[columnas]
    .sort_values('sucursales_provincia')
)

DIA — un ejemplo por cada código:


,sucursales_provincia,sucursales_localidad,sucursales_nombre
343,AR-A,SALTA,813 - SALTA
7,AR-B,BUENOS AIRES,9 - CIUDADELA
0,AR-C,CAPITAL FEDERAL,1 - VILLA DEL PARQUE
237,AR-E,ENTRE RIOS,305 - GUALEGUAYCHU
756,AR-J,JUJUY,809 - JUJUY
167,AR-S,SANTA FE,211 - VILLA CONSTITUCION
279,AR-W,CORRIENTES,602 - MANANTIALES DPTO S.ROQUE
394,AR-X,CORDOBA,1060 - VILLA MARIA
982,Buenos Aires,CABA,eCommerce



Carrefour — un ejemplo por cada código:


,sucursales_provincia,sucursales_localidad,sucursales_nombre
54,AR-A,Escobar,GELVES 530
3,AR-B,San Miguel,C.Tribulato 1279 (San Miguel)
0,AR-C,Ciudad Autónoma de Buenos Aires,Nazca 1944
69,AR-D,San Luis,San Luis
51,AR-E,Concordia,Concordia II
444,AR-F,La Rioja,La Rioja
404,AR-H,Resistencia,Resistencia II
358,AR-J,San Juan,San Juan
214,AR-K,San Fernando del Valle de Catamarca,Catamarca
506,AR-L,Santa Rosa,Santa Rosa


### 4.2 Hallazgos sobre inconsistencias en los códigos ISO

Durante la inspección de los valores únicos de `sucursales_provincia` en los datasets de DIA y Carrefour se detectaron inconsistencias que requieren verificación antes de reemplazar los códigos ISO por los nombres de las provincias.

- **DIA:** el código `AR-J` aparece asociado a la provincia de *Jujuy*.  
  Sin embargo, según la norma ISO 3166-2, `AR-J` corresponde a *San Juan*.  
  Es necesario comprobar si este error ocurre en un único registro o si DIA utiliza sistemáticamente `AR-J` para referirse a Jujuy.

- **Carrefour:** el código `AR-A` aparece asignado a una sucursal ubicada en *Buenos Aires*.  
  Según ISO 3166-2, `AR-A` corresponde a *Salta*.  
  Este caso también debe analizarse para determinar si se trata de un error aislado o de un patrón dentro del dataset.

Estas inconsistencias deben resolverse antes de realizar la conversión definitiva de códigos ISO a nombres de provincias, para evitar asignaciones incorrectas en el análisis geográfico.


In [299]:
# Provincias presentes en cada dataset
provincias_dia = set(sucursales_dia['sucursales_provincia'].unique())
provincias_carrefour = set(sucursales_carrefour['sucursales_provincia'].unique())

# Intersección: provincias donde ambas cadenas tienen presencia
provincias_coincidentes = provincias_dia.intersection(provincias_carrefour)

provincias_coincidentes


{'AR-A', 'AR-B', 'AR-C', 'AR-E', 'AR-J', 'AR-S', 'AR-W', 'AR-X'}

In [300]:
# Inspeccionamos las sucursales de DIA que pertenecen a la provincia 'AR-J' (Jujuy).
dia_ar_j = sucursales_dia[sucursales_dia['sucursales_provincia'] == 'AR-J']

dia_ar_j[['id_sucursal', 'sucursales_nombre', 'sucursales_localidad', 'sucursales_provincia']]

,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_provincia
756,809,809 - JUJUY,JUJUY,AR-J
759,823,823 - MONTERRICO,JUJUY,AR-J
828,1806,1806 - JUJUY,JUJUY,AR-J
829,1807,1807 - PALPALA,JUJUY,AR-J
830,1808,1808 - PERICO DEL CARMEN,JUJUY,AR-J
981,8018,8018 - JUJUY,JUJUY,AR-J


In [301]:
# Imputamos la provincia correcta para las sucursales de DIA que tienen el código 'AR-J', 
# cambiándolo a 'AR-Y' (código ISO correcto para Jujuy).
sucursales_dia.loc[
    sucursales_dia['sucursales_provincia'] == 'AR-J',
    'sucursales_provincia'
] = 'AR-Y'


### 4.3 Inconsistencia confirmada en DIA: uso de `AR-J` para Jujuy

Al analizar todas las sucursales de DIA que utilizan el código `AR-J`, se observa que **todas pertenecen a la provincia de Jujuy**.  
Sin embargo, según la norma ISO 3166-2, el código `AR-J` corresponde a **San Juan**, no a Jujuy.

Este hallazgo confirma que DIA utiliza sistemáticamente `AR-J` para referirse a Jujuy, lo cual constituye una codificación incorrecta respecto del estándar ISO.  
Además, dado que Carrefour sí utiliza `AR-J` correctamente para **San Juan**, esta inconsistencia implica que **San Juan no puede considerarse una provincia coincidente entre ambas cadenas**, reduciendo el conjunto de provincias relevantes para el análisis comparativo.



In [302]:
# Eliminamos 'AR-J' de la lista de provincias coincidentes, ya que no es un código ISO válido para Jujuy. 
provincias_coincidentes.discard('AR-J')
provincias_coincidentes

{'AR-A', 'AR-B', 'AR-C', 'AR-E', 'AR-S', 'AR-W', 'AR-X'}

## 5. Normalización de provincias relevantes para el análisis comparativo

Para garantizar que el análisis entre DIA y Carrefour se realice únicamente sobre información consistente y comparable, se decidió normalizar y corregir las provincias **solo dentro del conjunto de provincias donde ambas cadenas tienen presencia**.  
Este enfoque evita imputar o corregir provincias que no influyen en la comparación y permite concentrar el trabajo en los casos que realmente afectan el análisis.

El proceso se realizará **provincia por provincia**, inspeccionando cada código utilizado por ambas cadenas, detectando inconsistencias y corrigiendo únicamente las filas que pertenecen a las provincias coincidentes.  
De esta manera, cada provincia relevante quedará completamente normalizada antes de avanzar con la etapa de análisis.

A continuación se inspeccionan las provincias coincidentes `('AR-A', 'AR-B', 'AR-C', 'AR-E', 'AR-S', 'AR-W', 'AR-X')` y se corrigen sus inconsistencias una por una.

### 5.1 'AR-A' Carrefour y Dia

### 5.1.1 'AR-A' en Carrefour

Dado que en esta etapa del análisis se requiere repetir el mismo procedimiento para cada código ISO de provincia y para ambas cadenas (Carrefour y DIA), se definió una función auxiliar en el módulo `src/inspeccion.py`.  
Esta función centraliza la lógica de inspección y permite mantener el notebook más limpio, evitando duplicación de código y asegurando un flujo de trabajo más ordenado y profesional durante todo el proceso de normalización geográfica.

In [303]:
import sys
import os

# Agregar la carpeta raíz del proyecto al path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [304]:
from src.inspeccion import inspeccionar_provincia

In [305]:
# Inspeccionamos las sucursales de Carrefour que pertenecen a la provincia 'AR-A'.
inspeccionar_provincia(sucursales_carrefour, 'AR-A')

,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_provincia,sucursales_latitud,sucursales_longitud
54,187,GELVES 530,Escobar,AR-A,-34.343910,-58.789790
83,175,Salta VII,Salta,AR-A,-24.779652,-65.427437
280,112,Salta,Salta,AR-A,-24.789072,-65.413699
321,171,Salta III,Salta,AR-A,-24.781285,-65.401667
367,73,MAXI PINAMAR (73),Pinamar,AR-A,-37.109037,-56.893103
607,502,MAXI - JUJUY (502),Jujuy,AR-A,-24.187190,-65.303709


La inspección de todas las sucursales de Carrefour que utilizan el código `AR-A` revela que este valor aparece asignado a sucursales ubicadas en **tres provincias distintas**:

- Buenos Aires (Escobar, Pinamar)
- Salta (tres sucursales)
- Jujuy (una sucursal)

Según la norma ISO 3166-2, el código `AR-A` corresponde exclusivamente a la provincia de *Salta*.  
Sin embargo, Carrefour utiliza este código de manera inconsistente para referirse a sucursales ubicadas en provincias diferentes.

Con esto se confirma que los códigos ISO presentes en el dataset de Carrefour **no pueden utilizarse directamente como referencia geográfica confiable**.  
Antes de reemplazar los códigos ISO por los nombres de las provincias, será necesario corregir explícitamente estos casos para evitar asignaciones erróneas en el análisis regional.

Un hallazgo relevante es que entre las sucursales mal imputadas bajo `AR-A` se identificó una sucursal ubicada en la provincia de *Jujuy*.  
Este caso resulta especialmente significativo porque previamente se había detectado que DIA también imputaba incorrectamente a Jujuy bajo el código `AR-J`.  
Tras corregir ambas inconsistencias y asignar el código ISO correcto (`AR-Y`) en las dos cadenas, se incorporará a Jujuy como una provincia coincidente real dentro del conjunto de provincias coincidentes para el análisis comparativo.



In [306]:
# Corrección de AR-A en Carrefour según localidad
mapa_provincias_carrefour_ar_a = {
    'Escobar': 'AR-B',   # Buenos Aires
    'Pinamar': 'AR-B',   # Buenos Aires
    'Salta': 'AR-A',     # Salta (correcto)
    'Jujuy': 'AR-Y',     # Jujuy (este es el código ISO correcto)
}

sucursales_carrefour.loc[
    sucursales_carrefour['sucursales_provincia'] == 'AR-A',
    'sucursales_provincia'
] = sucursales_carrefour.loc[
    sucursales_carrefour['sucursales_provincia'] == 'AR-A',
    'sucursales_localidad'
].map(mapa_provincias_carrefour_ar_a)


Luego de esta imputación, Jujuy debería sumarse a las provincias coincidentes.

In [307]:
# Comprobamos nuevamente las provincias presentes en cada dataset después de las correcciones.
provincias_dia = set(sucursales_dia['sucursales_provincia'].unique())
provincias_carrefour = set(sucursales_carrefour['sucursales_provincia'].unique())

provincias_coincidentes = provincias_dia.intersection(provincias_carrefour)
provincias_coincidentes


{'AR-A', 'AR-B', 'AR-C', 'AR-E', 'AR-S', 'AR-W', 'AR-X', 'AR-Y'}

### Provincias coincidentes actualizadas

Tras corregir las imputaciones incorrectas en ambas cadenas —incluyendo la normalización de Jujuy (`AR-Y`) en DIA y Carrefour— el conjunto de provincias coincidentes queda actualizado.  
Estas provincias representan el universo geográfico donde ambas cadenas tienen presencia real y correctamente imputada, y serán utilizadas para el análisis comparativo.

Provincias coincidentes: `('AR-A', 'AR-B', 'AR-C', 'AR-E', 'AR-S', 'AR-W', 'AR-X', 'AR-Y')`



#### 5.1.2 'AR-A' en DIA

In [308]:
# Inspeccionamos las sucursales de Dia que pertenecen a la provincia 'AR-A'.
inspeccionar_provincia(sucursales_dia, 'AR-A')

,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_provincia,sucursales_latitud,sucursales_longitud
343,813,813 - SALTA,SALTA,AR-A,-24.849116,-65.43395
344,815,815 - SALTA,SALTA,AR-A,-24.797052,-65.398579
345,819,819 - CERRILLOS,SALTA,AR-A,-24.905302,-65.48719
346,820,820 - SALTA,SALTA,AR-A,-24.812077,-65.383161
347,821,821 - JOAQUIN V GONZALEZ,SALTA,AR-A,-24.7871267,-65.4177599
754,803,803 - SALTA,SALTA,AR-A,-24.7947076,-65.4322727
755,806,806 - SALTA,SALTA,AR-A,-24.808249,-65.399047
757,810,810 - SALTA,SALTA,AR-A,-24.769449,-65.443309
758,811,811 - SALTA,SALTA,AR-A,-24.7789448,-65.4045791
760,824,824 - SAN LORENZO,SALTA,AR-A,-24.74516,-65.4843


Al inspeccionar las sucursales de DIA que utilizan el código `AR-A`, se observa que todas las sucursales corresponden efectivamente a la provincia de *Salta*.  
A diferencia de Carrefour, DIA utiliza `AR-A` de manera consistente y alineada con la norma ISO 3166-2, sin mezclar localidades de otras provincias bajo este código.

Por lo tanto, no es necesario aplicar correcciones en este caso, y la provincia queda normalizada para el análisis comparativo.

### 5.2 'AR-B' Carrefour y Dia

#### 5.2.1 'AR-B' en Carrefour

In [309]:
# Inspeccionamos las sucursales de Carrefour que pertenecen a la provincia 'AR-B'.
inspeccionar_provincia(sucursales_carrefour, 'AR-B')

,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_provincia,sucursales_latitud,sucursales_longitud
3,755,C.Tribulato 1279 (San Miguel),San Miguel,AR-B,-34.541507,-58.713798
4,479,Avenida Eva Perón 4215,Merlo,AR-B,-34.690047,-58.689142
10,524,Bernardo Irigoyen 746,General Rodríguez,AR-B,-34.608190,-58.953570
18,8,La Plata,La Plata,AR-B,-34.893655,-57.996156
21,704,704 MAXI - EZPELETA (704),Ezpeleta,AR-B,-34.748715,-58.223592
...,...,...,...,...,...,...
694,1,San Isidro,San Isidro,AR-B,-34.491345,-58.589025
697,422,Murature 2169,Lomas de Zamora,AR-B,-34.707006,-58.438947
699,431,Av. Perón 3283,Victoria,AR-B,-34.453474,-58.538348
707,296,Olavarría II,Olavarría,AR-B,-36.908071,-60.314098


In [310]:
# Inspeccionamos los registros únicos de localidades para la provincia 'AR-B' en Carrefour.
carrefour_ar_b = sucursales_carrefour[sucursales_carrefour['sucursales_provincia'] == 'AR-B']

carrefour_ar_b['sucursales_localidad'].unique()

<ArrowStringArray>
[               'San Miguel',                     'Merlo',
         'General Rodríguez',                  'La Plata',
                  'Ezpeleta',             'Villa Adelina',
                 'Ituzaingó',               'Ramos Mejía',
                   'Quilmes',             'Vicente López',
           'Villa Ballester',       'Malvinas Argentinas',
                 'Maschwitz',                   'Escobar',
                     'Wilde',              'San Fernando',
                     'Pilar',           'Isidro Casanova',
            'Los Polvorines',                'San Martín',
                  'Martínez',     'Gregorio de Laferrere',
                     'Lanús',                    'Moreno',
                      'Glew',                     'Munro',
                   'Caseros',                 'Pergamino',
               'San Nicolás',                'La Matanza',
                 'San Justo',      'San Antonio de Padua',
              'Villa Celina',      'R

Al inspeccionar las localidades asociadas al código `AR-B` en Carrefour, se observa que la mayoría corresponden efectivamente a la provincia de Buenos Aires.  
Sin embargo, dentro del conjunto aparecen también las localidades **NEUQUEN** y **La Pampa**, que pertenecen a provincias distintas y no deberían estar imputadas bajo `AR-B`.

Este hallazgo confirma que el código `AR-B` está siendo utilizado de manera inconsistente en el dataset de Carrefour, mezclando sucursales de diferentes provincias bajo un mismo código ISO.  
Antes de proceder con la normalización, será necesario verificar si estos casos corresponden a homónimos dentro de Buenos Aires o si se trata de errores de imputación que deben corregirse explícitamente.


In [311]:
carrefour_ar_b[carrefour_ar_b['sucursales_localidad'].isin(['NEUQUEN', 'La Pampa'])]


,id_comercio,id_bandera,id_sucursal,sucursales_nombre,sucursales_tipo,sucursales_calle,sucursales_numero,sucursales_latitud,sucursales_longitud,sucursales_observaciones,sucursales_barrio,sucursales_codigo_postal,sucursales_localidad,sucursales_provincia,sucursales_lunes_horario_atencion,sucursales_martes_horario_atencion,sucursales_miercoles_horario_atencion,sucursales_jueves_horario_atencion,sucursales_viernes_horario_atencion,sucursales_sabado_horario_atencion,sucursales_domingo_horario_atencion
346,10,4,68,Homero Manzi (Ruta 17) y Ruta 22,Hipermercado,HOMERO MANZI Y RUTA 22,0,-38.929334,-69.207412,NaN,NaN,8318,NEUQUEN,AR-B,08:30 a 20:00,08:30 a 20:00,08:30 a 20:00,08:30 a 20:00,08:30 a 20:00,08:30 a 20:00,09:00 a 20:00
644,10,4,67,Av Agustín Spinetto 551 (esquina Neuquén),Hipermercado,AV AGUSTIN SPINETTO,551,-36.610397,-64.281612,NaN,NaN,6302,La Pampa,AR-B,08:30 a 20:00,08:30 a 20:00,08:30 a 20:00,08:30 a 20:00,08:30 a 20:00,08:30 a 20:00,09:00 a 20:00


La verificación de las direcciones y localidades confirma que estas sucursales corresponden efectivamente a las provincias de *Neuquén* y *La Pampa*, y no a Buenos Aires.

Este resultado demuestra que el código `AR-B` está siendo utilizado de manera incorrecta para imputar sucursales ubicadas en otras provincias.  
Por lo tanto, será necesario corregir explícitamente estas imputaciones antes de avanzar con la normalización y el análisis comparativo.


In [312]:
# Corrección de localidades mal imputadas dentro de AR-B
mapa_provincias_carrefour_ar_b = {
    'NEUQUEN': 'AR-Q',   # Neuquén
    'La Pampa': 'AR-L',  # La Pampa
}

In [313]:
# Imputamos la provincia correcta para las sucursales de Carrefour.
mask_ar_b = (
    (sucursales_carrefour['sucursales_provincia'] == 'AR-B') &
    (sucursales_carrefour['sucursales_localidad'].isin(
        mapa_provincias_carrefour_ar_b.keys()
    ))
)

sucursales_carrefour.loc[
    mask_ar_b,
    'sucursales_provincia'
] = sucursales_carrefour.loc[
    mask_ar_b,
    'sucursales_localidad'
].map(mapa_provincias_carrefour_ar_b)


In [314]:
# Comprobamos si fueron correctamente imputadas las provincias para las localidades 'NEUQUEN' y 'La Pampa'.
sucursales_carrefour[
    sucursales_carrefour['sucursales_localidad'].isin(['NEUQUEN', 'La Pampa'])
][['id_sucursal','sucursales_nombre','sucursales_localidad','sucursales_provincia']]


,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_provincia
346,68,Homero Manzi (Ruta 17) y Ruta 22,NEUQUEN,AR-Q
644,67,Av Agustín Spinetto 551 (esquina Neuquén),La Pampa,AR-L


#### 5.2.2 'AR-B' en Dia

In [315]:
# Inspeccionamos las sucursales de Dia que pertenecen a la provincia 'AR-B'.
dia_ar_b = inspeccionar_provincia(sucursales_dia, 'AR-B')
dia_ar_b

,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_provincia,sucursales_latitud,sucursales_longitud
7,9,9 - CIUDADELA,BUENOS AIRES,AR-B,-34.6402735,-58.5418243
8,10,10 - CASEROS,BUENOS AIRES,AR-B,-34.601775,-58.5618167
11,14,14 - SAN MARTIN,BUENOS AIRES,AR-B,-34.5738643,-58.5326316
13,19,19 - SAN JUSTO,BUENOS AIRES,AR-B,-34.6830345,-58.5553009
16,23,23 - RAFAEL CASTILLO,BUENOS AIRES,AR-B,-34.6997046,-58.6247299
...,...,...,...,...,...,...
964,5456,5456 - SAN MARTIN,BUENOS AIRES,AR-B,-34.594044,-58.537335
965,5458,5458 - MORENO,BUENOS AIRES,AR-B,-34.636677,-58.802593
966,5461,5461 - CIUDADELA,BUENOS AIRES,AR-B,-34.634849,-58.542463
967,5464,5464 - LUJAN,BUENOS AIRES,AR-B,-34.573355,-59.115607


In [316]:
dia_ar_b['sucursales_localidad'].unique()

<ArrowStringArray>
['BUENOS AIRES']
Length: 1, dtype: str

Al inspeccionar las sucursales de DIA imputadas bajo el código `AR-B`, se observa que todas las localidades asociadas corresponden efectivamente a la provincia de Buenos Aires.  
A diferencia de Carrefour, DIA utiliza el código `AR-B` de manera consistente y alineada con la norma ISO 3166-2, sin mezclar sucursales de otras provincias bajo este identificador.

No se detectaron inconsistencias ni imputaciones erróneas en esta provincia, por lo que `AR-B` queda normalizada y lista para el análisis comparativo.


### 5.3 'AR-C' en Dia y Carrefour

#### 5.3.1 'AR-C' en Carrefour

In [317]:
# Inspeccionamos las sucursales de Carrefour que pertenecen a la provincia 'AR-C'.
carrefour_ar_c = inspeccionar_provincia(sucursales_carrefour, 'AR-C')

carrefour_ar_c

,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_provincia,sucursales_latitud,sucursales_longitud
0,365,Nazca 1944,Ciudad Autónoma de Buenos Aires,AR-C,-34.612145,-58.480997
1,408,Perú 916,Ciudad Autónoma de Buenos Aires,AR-C,-34.618745,-58.374322
2,504,Maxi Cacheuta Cba (504),Cacheuta,AR-C,-31.460585,-64.215363
5,245,Avenida San Martín 2999,Ciudad Autónoma de Buenos Aires,AR-C,-34.601957,-58.467059
6,423,Av. Castañares 4854,Ciudad Autónoma de Buenos Aires,AR-C,-34.667069,-58.472973
...,...,...,...,...,...,...
725,258,Villa Pueyrredon,Ciudad Autónoma de Buenos Aires,AR-C,-34.585933,-58.501809
726,344,M. T. de Alvear 2251,Ciudad Autónoma de Buenos Aires,AR-C,-34.596862,-58.399571
727,593,Entre Ríos 1192,Ciudad Autónoma de Buenos Aires,AR-C,-34.622603,-58.391341
728,791,BEIRO 3342,Ciudad Autónoma de Buenos Aires,AR-C,-34.598084,-58.500119


La inspección de las sucursales imputadas bajo el código `AR-C` muestra un volumen elevado de registros, lo cual es esperable dado que este código corresponde a la Ciudad Autónoma de Buenos Aires (CABA).  
Para garantizar que el análisis se realice únicamente sobre sucursales efectivamente ubicadas en esta jurisdicción, se aplicará un filtro que excluya todas las localidades distintas de **'Ciudad Autónoma de Buenos Aires'**, permitiendo así depurar posibles imputaciones incorrectas dentro de este código ISO.


In [318]:
carrefour_ar_c = sucursales_carrefour[
    (sucursales_carrefour['sucursales_provincia'] == 'AR-C') &
    (sucursales_carrefour['sucursales_localidad'] != 'Ciudad Autónoma de Buenos Aires')
]

carrefour_ar_c[['id_sucursal', 'sucursales_nombre', 'sucursales_localidad', 'sucursales_provincia', 'sucursales_latitud', 'sucursales_longitud']]

,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_provincia,sucursales_latitud,sucursales_longitud
2,504,Maxi Cacheuta Cba (504),Cacheuta,AR-C,-31.460585,-64.215363
7,503,Maxi San Vicente (503),San Vicente,AR-C,-34.999979,-58.397284
20,867,9 de Julio 156,Córdoba,AR-C,-31.090725,-64.485310
43,270,Jose Ingenieros 59,Córdoba,AR-C,-31.422513,-64.496978
57,707,AVDA JUAN B.JUSTO 4775 (5000),Córdoba,AR-C,-31.367973,-64.177617
67,172,San Juan 1128,Córdoba,AR-C,-32.890284,-68.837131
163,45,25 DE MAYO 1665,Trelew,AR-C,-43.258515,-65.329191
220,224,Ambrosio Olmos 929,Córdoba,AR-C,-31.431692,-64.191582
221,876,Luis Agote 2092,Córdoba,AR-C,-31.426419,-64.215643
237,860,Lopez Y Planes 265,Córdoba,AR-C,-30.982034,-64.491897


Al filtrar las sucursales imputadas bajo el código `AR-C` que no pertenecen a la localidad *Ciudad Autónoma de Buenos Aires*, se observa que una gran cantidad de registros están asignados incorrectamente.  
La revisión de las coordenadas (latitud y longitud) confirma que la mayoría de estas sucursales corresponden a la provincia de **Córdoba**, incluyendo una sucursal ubicada en *Cacheuta*, cuya georreferencia también coincide con dicha provincia.  
Asimismo, se detecta una sucursal ubicada en **San Luis**, y varias localidades que pertenecen a la provincia de Buenos Aires.

Este hallazgo evidencia que el código `AR-C` está siendo utilizado de manera inconsistente en el dataset de Carrefour, mezclando sucursales de distintas provincias bajo el identificador correspondiente a la Ciudad Autónoma de Buenos Aires.  
Será necesario corregir estas imputaciones antes de avanzar con la normalización y el análisis comparativo.

In [319]:
# Corrección de localidades mal imputadas dentro de AR-C
mapa_provincias_carrefour_ar_c = {
    'Córdoba': 'AR-X',   # Córdoba
    'Cacheuta': 'AR-X',  # Córdoba
    'San Luis': 'AR-D',    # San Luis
}

In [320]:
# Imputamos la provincia correcta para las sucursales de Carrefour.
mask_ar_c = (
    (sucursales_carrefour['sucursales_provincia'] == 'AR-C') &
    (sucursales_carrefour['sucursales_localidad'].isin(
        mapa_provincias_carrefour_ar_c.keys()
    ))
)

sucursales_carrefour.loc[
    mask_ar_c,
    'sucursales_provincia'
] = sucursales_carrefour.loc[
    mask_ar_c,
    'sucursales_localidad'
].map(mapa_provincias_carrefour_ar_c)

In [321]:
carrefour_ar_c_proc = sucursales_carrefour[
    (sucursales_carrefour['sucursales_provincia'] == 'AR-C') &
    (sucursales_carrefour['sucursales_localidad'] != 'Ciudad Autónoma de Buenos Aires')
]

carrefour_ar_c_proc[['id_sucursal', 'sucursales_nombre', 'sucursales_localidad', 'sucursales_provincia', 'sucursales_latitud', 'sucursales_longitud']]

,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_provincia,sucursales_latitud,sucursales_longitud
7,503,Maxi San Vicente (503),San Vicente,AR-C,-34.999979,-58.397284
163,45,25 DE MAYO 1665,Trelew,AR-C,-43.258515,-65.329191
257,701,LAFERRERE,Laferrere,AR-C,-34.754617,-58.612940
319,501,MAXI - Virrey del Pino (501),Virrey del Pino,AR-C,-34.837127,-58.650749
409,71,Maxi Cañuelas (71),Cañuelas,AR-C,-35.037370,-58.758240
414,40,AV. EVA PERON 2679,Temperley,AR-C,-34.763279,-58.364638
446,702,CLAYPOLE,Claypole,AR-C,-34.822169,-58.366915
604,72,MAXI CHIVILCOY (72),Chivilcoy,AR-C,-34.919834,-59.988270


Luego de corregir las imputaciones erróneas correspondientes a Córdoba, Cacheuta y San Luis, se realizó un segundo filtrado sobre las sucursales que permanecían bajo el código `AR-C` pero cuya localidad no coincide con *Ciudad Autónoma de Buenos Aires*.  
La inspección de estas filas revela que Carrefour continúa utilizando `AR-C` de manera incorrecta para imputar sucursales ubicadas en diversas localidades de la provincia de Buenos Aires, tales como San Vicente, Laferrere, Virrey del Pino, Cañuelas, Temperley, Claypole y Chivilcoy.  

Además, se detectó un caso adicional en **Trelew**, cuya georreferencia corresponde a la provincia de *Chubut*, lo que confirma otro error de imputación dentro de este código.

Este hallazgo final demuestra que el código `AR-C` presenta múltiples inconsistencias y requiere una corrección para garantizar una normalización geográfica confiable antes del análisis comparativo.

In [322]:
# Definimos las coordenadas geográficas aproximadas para las provincias de Buenos Aires, CABA y Chubut,
# para identificar sucursales que puedan estar mal imputadas en la provincia 'AR-C'.
bsas_ubi = (
    (carrefour_ar_c_proc['sucursales_latitud'] >= -39.0) &
    (carrefour_ar_c_proc['sucursales_latitud'] <= -33.0) &
    (carrefour_ar_c_proc['sucursales_longitud'] >= -63.0) &
    (carrefour_ar_c_proc['sucursales_longitud'] <= -57.0)
)
caba_ubi_2 = (
    (carrefour_ar_c_proc['sucursales_latitud'] >= -34.70) &
    (carrefour_ar_c_proc['sucursales_latitud'] <= -34.55) &
    (carrefour_ar_c_proc['sucursales_longitud'] >= -58.55) &
    (carrefour_ar_c_proc['sucursales_longitud'] <= -58.35)
)
chubut_ubi = (
    (carrefour_ar_c_proc['sucursales_latitud'] >= -46.0) &
    (carrefour_ar_c_proc['sucursales_latitud'] <= -41.0) &
    (carrefour_ar_c_proc['sucursales_longitud'] >= -72.0) &
    (carrefour_ar_c_proc['sucursales_longitud'] <= -65.0)
)


In [323]:
# CABA
sucursales_carrefour.loc[carrefour_ar_c_proc.index[caba_ubi_2], 'sucursales_provincia'] = 'AR-C'

# Buenos Aires
sucursales_carrefour.loc[carrefour_ar_c_proc.index[bsas_ubi], 'sucursales_provincia'] = 'AR-B'

# Chubut
sucursales_carrefour.loc[carrefour_ar_c_proc.index[chubut_ubi], 'sucursales_provincia'] = 'AR-U'

In [324]:
# Verificamos que se hayan imputado correctamente las provincias.
sucursales_carrefour.loc[carrefour_ar_c_proc.index][
    ['id_sucursal','sucursales_localidad','sucursales_provincia']
]

,id_sucursal,sucursales_localidad,sucursales_provincia
7,503,San Vicente,AR-B
163,45,Trelew,AR-U
257,701,Laferrere,AR-B
319,501,Virrey del Pino,AR-B
409,71,Cañuelas,AR-B
414,40,Temperley,AR-B
446,702,Claypole,AR-B
604,72,Chivilcoy,AR-B


#### 5.3.2 'AR-C' en Dia.

In [325]:
# Inspeccionamos las sucursales de Dia que pertenecen a la provincia 'AR-C'.
dia_ar_c = inspeccionar_provincia(sucursales_dia, 'AR-C')

dia_ar_c

,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_provincia,sucursales_latitud,sucursales_longitud
0,1,1 - VILLA DEL PARQUE,CAPITAL FEDERAL,AR-C,-34.5971373,-58.4976429
1,3,3 - BALVANERA,CAPITAL FEDERAL,AR-C,-34.6173202,-58.3916913
2,4,4 - PARQUE PATRICIOS,CAPITAL FEDERAL,AR-C,-34.6369605,-58.4045551
3,5,5 - SAN CRISTÓBAL,CAPITAL FEDERAL,AR-C,-34.6236734,-58.4014697
4,6,6 - FLORES,CAPITAL FEDERAL,AR-C,-34.6289482,-58.4642662
...,...,...,...,...,...,...
817,1531,1531 - PARQUE PATRICIOS,CAPITAL FEDERAL,AR-C,-34.6326213,-58.4061362
818,1532,1532 - MATADEROS,CAPITAL FEDERAL,AR-C,-34.6479782,-58.5063272
819,1533,1533 - MATADEROS,CAPITAL FEDERAL,AR-C,-34.6651686,-58.5033475
821,1538,1538 - VILLA LURO,CAPITAL FEDERAL,AR-C,-34.6278546,-58.5156425


In [326]:
# Verificamos que todas las localidades sean CAPITAL FEDERAL
# tal como se observa en la línea anterior de código.
sucursales_dia[
    sucursales_dia['sucursales_provincia'] == 'AR-C'
]['sucursales_localidad'].unique()


<ArrowStringArray>
['CAPITAL FEDERAL']
Length: 1, dtype: str

La verificación de las sucursales imputadas bajo el código `AR-C` muestra que todas las tiendas de DIA asociadas a este identificador corresponden efectivamente a **CAPITAL FEDERAL**.  
No se detectaron localidades externas ni inconsistencias en la imputación, por lo que la provincia `AR-C` queda validada y correctamente normalizada para esta cadena.

### 5.4 'AR-E' en Dia y Carrefour

#### 5.4.1 'AR-E' en Carrefour.

In [327]:
# Inspeccionamos las sucursales de Carrefour que pertenecen a la provincia 'AR-E'.
carrefour_ar_e = inspeccionar_provincia(sucursales_carrefour, 'AR-E')
carrefour_ar_e

,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_provincia,sucursales_latitud,sucursales_longitud
51,138,Concordia II,Concordia,AR-E,-31.396866,-58.018603
147,264,Paraná (México),Paraná,AR-E,-31.725619,-60.533681
175,233,Concordia,Concordia,AR-E,-31.368882,-58.011377
201,263,Parana Rocamora,Paraná,AR-E,-31.754009,-60.511242
300,244,H.YRIGOYEN 223/63,Paraná,AR-E,-31.737677,-60.524753
415,266,AV.D. BOSCO ESQ. B. DE SUIPACHA,Paraná,AR-E,-31.733841,-60.507686
470,240,Gualeguaychú,Gualeguaychu,AR-E,-32.991736,-58.521853
495,181,Parana Parque,Paraná,AR-E,-31.722868,-60.529162
513,262,Parana J M Paz,Paraná,AR-E,-31.741747,-60.541261
556,243,Paraná (San Juan),Paraná,AR-E,-31.726740,-60.524157


La inspección de las sucursales imputadas bajo el código `AR-E` muestra que todas las tiendas de Carrefour se encuentran correctamente asignadas a la provincia de Entrer Ríos.

#### 5.4.2 'AR-E' en Dia.

In [328]:
# Inspeccionamos las sucursales de Dia que pertenecen a la provincia 'AR-E'.
dia_ar_e = inspeccionar_provincia(sucursales_dia, 'AR-E')
dia_ar_e

,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_provincia,sucursales_latitud,sucursales_longitud
237,305,305 - GUALEGUAYCHU,ENTRE RIOS,AR-E,-33.0071111,-58.5119321
245,319,319 - COLON,ENTRE RIOS,AR-E,-32.2252713,-58.1454569
246,320,320 - NOGOYA,ENTRE RIOS,AR-E,-32.397116,-59.792151
248,323,323 - VICTORIA,ENTRE RIOS,AR-E,-32.6220245,-60.1570702
250,325,325 - ROSARIO DEL TALA,ENTRE RIOS,AR-E,-32.301798,-59.138836
...,...,...,...,...,...,...
971,6047,6047 - VILLAGUAY,ENTRE RIOS,AR-E,-31.8696614,-59.0284795
972,6058,6058 - SAN SALVADOR,ENTRE RIOS,AR-E,-31.62669,-58.50777
973,6061,6061 - GENERAL GALARZA,ENTRE RIOS,AR-E,-32.723682,-59.396034
974,6063,6063 - GUALEGUAYCHU,ENTRE RIOS,AR-E,-33.006972,-58.537419


In [329]:
# Verificamos que todas las localidades sean ENTRE RIOS
# tal como se observa en la línea anterior de código.
sucursales_dia[
    sucursales_dia['sucursales_provincia'] == 'AR-E'
]['sucursales_localidad'].unique()

<ArrowStringArray>
['ENTRE RIOS']
Length: 1, dtype: str

La verificación de las sucursales imputadas bajo el código `AR-E` muestra que todas las tiendas de Dia asociadas a este identificador corresponden efectivamente a **Entre Ríos**.  
No se detectaron localidades externas ni inconsistencias en la imputación, por lo que la provincia `AR-E` queda validada y correctamente normalizada para esta cadena.

### 5.5 'AR-S' en Dia y Carrefour

#### 5.5.1 'AR-S' en Carrefour

In [330]:
# Inspeccionamos las sucursales de Carrefour que pertenecen a la provincia 'AR-S'.
carrefour_ar_s = inspeccionar_provincia(sucursales_carrefour, 'AR-S')
carrefour_ar_s

,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_provincia,sucursales_latitud,sucursales_longitud
241,33,Rosario Sur,Rosario,AR-S,-32.970122,-60.628560
318,32,Rosario Centro,Rosario,AR-S,-32.951847,-60.670442
383,259,Venado Tuerto,Venado Tuerto,AR-S,-33.744187,-61.982983
472,18,Rosario Fisherton,Rosario,AR-S,-32.904529,-60.727936
518,268,Rosario Pueyrredón,Rosario,AR-S,-32.942435,-60.657028
590,41,Rosario Village,Rosario,AR-S,-32.934019,-60.702038


#### 5.5.2 'AR-S' en Dia.

In [331]:
# Inspeccionamos las sucursales de Dia que pertenecen a la provincia 'AR-S'.
dia_ar_s = inspeccionar_provincia(sucursales_dia, 'AR-S')
dia_ar_s

,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_provincia,sucursales_latitud,sucursales_longitud
167,211,211 - VILLA CONSTITUCION,SANTA FE,AR-S,-33.225442,-60.332344
178,225,225 - FIRMAT,SANTA FE,AR-S,-33.459792,-61.488451
188,244,244 - VENADO TUERTO,SANTA FE,AR-S,-33.75,-61.965641
219,282,282 - ARROYO SECO,SANTA FE,AR-S,-33.156124,-60.512711
270,438,438 - FUNES,SANTA FE,AR-S,-32.920926,-60.812213
393,1059,1059 - SANTA FE,SANTA FE,AR-S,-31.649562,-60.7106075
399,1068,1068 - SANTO TOME,SANTA FE,AR-S,-31.6653404,-60.7653809
924,6052,6052 - VILLA CAÑAS,SANTA FE,AR-S,-34.002525,-61.609768
969,6017,6017 - VILLA CONSTITUCION,SANTA FE,AR-S,-33.2428923,-60.3179436


La verificación de las sucursales imputadas bajo el código `AR-S` muestra que tanto Carrefour como DIA tienen todas sus tiendas correctamente asignadas a **Santa Fe**.  
No se detectaron localidades externas ni inconsistencias en la imputación, por lo que la provincia `AR-S` queda validada y correctamente normalizada en ambas cadenas.


### 5.6 'AR-W' en Carrefour y Dia

#### 5.6.1 'AR-W' en Carrefour

In [332]:
# Inspeccionamos las sucursales de Carrefour que pertenecen a la provincia 'AR-W'.
carrefour_ar_w = inspeccionar_provincia(sucursales_carrefour, 'AR-W')
carrefour_ar_w

,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_provincia,sucursales_latitud,sucursales_longitud
209,159,Corrientes,Corrientes,AR-W,-27.477092,-58.815872
328,110,Corrientes Junin,Corrientes,AR-W,-27.467798,-58.834021


#### 5.6.2 'AR-W' en Dia

In [333]:
# Inspeccionamos las sucursales de Dia que pertenecen a la provincia 'AR-W'.
dia_ar_w = inspeccionar_provincia(sucursales_dia, 'AR-W')
dia_ar_w

,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_provincia,sucursales_latitud,sucursales_longitud
279,602,602 - MANANTIALES DPTO S.ROQUE,CORRIENTES,AR-W,-28.574823,-58.710841
280,604,604 - EMPEDRADO,CORRIENTES,AR-W,-27.954064,-58.808406
284,610,610 - CASUARINAS DPTO SALADAS,CORRIENTES,AR-W,-28.254287,-58.628393
293,623,623 - CORRIENTES ESTAFETA N 1,CORRIENTES,AR-W,-27.466137,-58.788067
310,651,651 - MERCEDES,CORRIENTES,AR-W,-29.180501,-58.092937
396,1063,1063 - MONTE CASEROS,CORRIENTES,AR-W,-30.250668,-57.63942
397,1064,1064 - CURUZU-CUATIA,CORRIENTES,AR-W,-29.791304,-58.055256
400,1069,1069 - GOYA,CORRIENTES,AR-W,-29.1452083,-59.2625031
662,575,575 - CURUZU-CUATIA,CORRIENTES,AR-W,-29.7794731,-58.073937
667,582,582 - CRUCECITAS SANTA LUCIA,CORRIENTES,AR-W,-28.986113,-59.10309


La verificación de las sucursales imputadas bajo el código `AR-W` confirma que tanto Carrefour como DIA tienen todas sus tiendas correctamente asignadas a **Corrientes**.  
No se detectaron localidades externas ni inconsistencias en la imputación, por lo que la provincia `AR-W` queda validada y correctamente normalizada en ambas cadenas.


### 5.7 'AR-X' en Carrefour y Dia

#### 5.7.1 'AR-X' en Carrefour

In [334]:
# Inspeccionamos las sucursales de Carrefour que pertenecen a la provincia 'AR-X'.
carrefour_ar_x = inspeccionar_provincia(sucursales_carrefour, 'AR-X')
carrefour_ar_x

,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_provincia,sucursales_latitud,sucursales_longitud
2,504,Maxi Cacheuta Cba (504),Cacheuta,AR-X,-31.460585,-64.215363
17,847,Avenida Edén 499,Córdoba,AR-X,-31.092842,-64.481358
20,867,9 de Julio 156,Córdoba,AR-X,-31.090725,-64.485310
27,789,DUARTE Q. 1851,Córdoba,AR-X,-31.411817,-64.209985
42,764,Av. San Lorenzo 480/84,Córdoba,AR-X,-31.425267,-64.179609
...,...,...,...,...,...,...
674,873,Derqui 33,Córdoba,AR-X,-31.425366,-64.187647
677,853,Avenida Colón 636,Córdoba,AR-X,-31.410798,-64.191844
696,793,SAN JUAN 835,Córdoba,AR-X,-31.417677,-64.196820
702,770,Ituzaingo 7514,Córdoba,AR-X,-31.423132,-64.184088


In [335]:
# Verificamos que todas las localidades sean de la provincia de Córdoba
# tal como se observa en la línea anterior de código.
sucursales_carrefour[
    sucursales_carrefour['sucursales_provincia'] == 'AR-X'
]['sucursales_localidad'].unique()

<ArrowStringArray>
[           'Cacheuta',             'Córdoba', 'Quintas de Arguello',
     'Jardin Espinoza', 'Villa Allende Lomas',          'Río Cuarto',
       'Villa Urquiza',          'Alto Verde']
Length: 8, dtype: str

Para esta provincia realizamos una inspección focalizada sobre las sucursales imputadas como `AR-X` cuya localidad **no coincide explícitamente con “Córdoba”**.  
El objetivo es revisar estos casos puntuales con mayor detalle —incluyendo la comparación de latitud y longitud— para detectar posibles anomalías geográficas o inconsistencias de imputación dentro del conjunto asignado a la provincia.

In [336]:
# Inspeccionamos específicamente las sucursales imputadas como 'AR-X' cuya localidad no coincide con 'Córdoba'. 
# Comparando también latitud y longitud para detectar posibles anomalías geográficas.
carrefour_ar_x_proc = sucursales_carrefour[
    (sucursales_carrefour['sucursales_provincia'] == 'AR-X') &
    (sucursales_carrefour['sucursales_localidad'] != 'Córdoba')
]

carrefour_ar_x_proc[['id_sucursal', 'sucursales_nombre', 'sucursales_localidad', 'sucursales_provincia', 'sucursales_latitud', 'sucursales_longitud']]

,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_provincia,sucursales_latitud,sucursales_longitud
2,504,Maxi Cacheuta Cba (504),Cacheuta,AR-X,-31.460585,-64.215363
127,50,Córdoba Recta Martinolli,Quintas de Arguello,AR-X,-31.346646,-64.269297
279,24,Córdoba Jardín,Jardin Espinoza,AR-X,-31.455534,-64.166095
401,54,Córdoba Villa Allende,Villa Allende Lomas,AR-X,-31.300019,-64.276462
454,119,Río Cuarto,Río Cuarto,AR-X,-33.125271,-64.349014
463,9,Córdoba Colon,Villa Urquiza,AR-X,-31.393915,-64.242835
647,295,Córdoba Granaderos,Alto Verde,AR-X,-31.374529,-64.213855


La inspección detallada de las sucursales imputadas bajo el código `AR-X` muestra que todas las tiendas analizadas corresponden efectivamente a la provincia de **Córdoba**.  
No se detectaron anomalías en las localidades ni en las coordenadas geográficas, por lo que la imputación de `AR-X` queda validada correctamente para esta cadena.


#### 5.7.2 'AR-X' en Dia

In [337]:
# Inspeccionamos las sucursales de Dia que pertenecen a la provincia 'AR-X'.
dia_ar_x = inspeccionar_provincia(sucursales_dia, 'AR-X')
dia_ar_x

,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_provincia,sucursales_latitud,sucursales_longitud
394,1060,1060 - VILLA MARIA,CORDOBA,AR-X,-32.4078411,-63.2485416
412,1513,1513 - HERNANDO,CORDOBA,AR-X,-32.426197,-63.738309
413,1515,1515 - ONCATIVO,CORDOBA,AR-X,-31.9151891,-63.6792919


La verificación de las sucursales imputadas bajo el código `AR-X` confirma que tanto Carrefour como DIA tienen todas sus tiendas correctamente asignadas a **Córdoba**.  
No se detectaron localidades externas ni inconsistencias en la imputación, por lo que la provincia `AR-X` queda validada y correctamente normalizada en ambas cadenas.

### 5.8 'AR-Y' en Carrefour y Dia

#### 5.8.1 'AR-Y' en Carrefour

In [338]:
# Inspeccionamos las sucursales de Carrefour que pertenecen a la provincia 'AR-Y'.
carrefour_ar_y = inspeccionar_provincia(sucursales_carrefour, 'AR-Y')
carrefour_ar_y

,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_provincia,sucursales_latitud,sucursales_longitud
607,502,MAXI - JUJUY (502),Jujuy,AR-Y,-24.18719,-65.303709


#### 5.8.2 'AR-Y' en Dia

In [339]:
# Inspeccionamos las sucursales de Dia que pertenecen a la provincia 'AR-Y'.
dia_ar_y = inspeccionar_provincia(sucursales_dia, 'AR-Y')
dia_ar_y

,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_provincia,sucursales_latitud,sucursales_longitud
756,809,809 - JUJUY,JUJUY,AR-Y,-24.194392,-65.289649
759,823,823 - MONTERRICO,JUJUY,AR-Y,-24.443343,-65.1674
828,1806,1806 - JUJUY,JUJUY,AR-Y,-24.19658,-65.309281
829,1807,1807 - PALPALA,JUJUY,AR-Y,-24.252255,-65.205416
830,1808,1808 - PERICO DEL CARMEN,JUJUY,AR-Y,-24.377394,-65.118997
981,8018,8018 - JUJUY,JUJUY,AR-Y,-24.387603,-65.264866


La verificación de las sucursales imputadas bajo el código `AR-Y` confirma que tanto Carrefour como DIA tienen todas sus tiendas correctamente asignadas a **Jujuy**.  
Las imputaciones coinciden con las localidades declaradas y no requieren ajustes adicionales, por lo que `AR-Y` queda validado en ambas cadenas.


### Finalización de la normalización de provincias coincidentes

Con la validación completa de las provincias coincidentes entre Carrefour y DIA, se confirma que todas las sucursales pertenecientes a este grupo quedaron correctamente normalizadas.  
En los casos donde se detectaron inconsistencias, las imputaciones fueron revisadas y corregidas provincia por provincia, garantizando que el conjunto final refleje la ubicación geográfica real de cada tienda.  
Esta etapa queda finalizada y el análisis puede continuar con las provincias no coincidentes o con la siguiente fase del estudio.



### 6. Unificación de `sucursales_provincia` con nombres de provincias

Hasta este punto, cada sucursal conserva el código ISO de su provincia (`AR-X`, `AR-S`, etc.), pero para análisis posteriores —especialmente tablas, gráficos y comparaciones entre cadenas— resulta más práctico trabajar con **nombres normalizados de provincias**.

En este paso construimos un diccionario que mapea cada código ISO a su nombre correspondiente, aplicando además una normalización adicional:
- conversión a mayúsculas,
- eliminación de tildes para evitar problemas en visualizaciones,
- ajustes manuales en casos específicos (`AR-V` y `AR-C`) para mantener consistencia con los datasets.

Luego aplicamos esta conversión tanto a DIA como a Carrefour, generando una columna final `provincia` ya unificada y lista para análisis.


In [340]:
from unidecode import unidecode

In [341]:
iso_a_nombre = dict(zip(provincias['iso_id'], provincias['nombre'].str.upper().apply(unidecode)))
iso_a_nombre['AR-V'] = 'TIERRA DEL FUEGO'  # el nombre oficial es muy largo para tablas/gráficos
iso_a_nombre['AR-C'] = 'CABA'  # para mantener consistencia con el dataset de Carrefour

def normalizar_provincia(valor):
    return iso_a_nombre.get(valor, valor)  # si ya es un código, lo convierte; si ya es nombre, lo deja

sucursales_dia['provincia'] = sucursales_dia['sucursales_provincia'].apply(normalizar_provincia)
sucursales_carrefour['provincia'] = sucursales_carrefour['sucursales_provincia'].apply(normalizar_provincia)

# Control: cada provincia debe aparecer una sola vez
print(sucursales_carrefour['provincia'].value_counts())

provincia
CABA                371
BUENOS AIRES        168
CORDOBA              82
MENDOZA              45
ENTRE RIOS           12
CHUBUT                7
SALTA                 6
SANTA FE              6
TUCUMAN               4
SANTA CRUZ            3
TIERRA DEL FUEGO      3
SAN LUIS              3
NEUQUEN               3
CATAMARCA             3
LA PAMPA              3
FORMOSA               2
RIO NEGRO             2
CORRIENTES            2
CHACO                 2
SAN JUAN              1
LA RIOJA              1
JUJUY                 1
Name: count, dtype: int64


La normalización de provincias quedó aplicada correctamente en ambas cadenas.  
Cada sucursal ahora cuenta con un nombre de provincia unificado, sin tildes y consistente entre DIA y Carrefour, lo que permite continuar el análisis sin divergencias entre códigos ISO, nombres largos o variantes ortográficas.

Los controles con (`value_counts`) muestran que la conversión se realizó de manera uniforme y que cada provincia aparece bajo una única denominación, cumpliendo el objetivo de este punto.


### 6.1 Verificación de consistencia geográfica en Carrefour

Durante la imputación de provincias en el dataset de Carrefour se detectaron varias inconsistencias entre la provincia declarada y la ubicación real de las sucursales.  
Para asegurar que el análisis posterior se base en información territorial confiable, se realiza una verificación adicional utilizando **filtros geográficos aproximados** utilizados para detectar posibles inconsistencias entre la provincia declarada y las coordenadas de la sucursal características de cada provincia.

El objetivo de este control es identificar sucursales cuya ubicación geográfica no coincide con la provincia asignada, ya sea por errores en el dataset original o por imputaciones previas que requieran revisión.  
Estos rangos no representan límites administrativos oficiales y se utilizan exclusivamente como mecanismo de detección de casos que requieren revisión.

In [342]:
# Se utilizan rangos aproximados de latitud y longitud para
# detectar sucursales cuya ubicación podría no coincidir con
# la provincia declarada.

# Buenos Aires - AR-B
bsas_ubi = (
    (sucursales_carrefour['sucursales_latitud'] >= -41.05) &
    (sucursales_carrefour['sucursales_latitud'] <= -33.25) &
    (sucursales_carrefour['sucursales_longitud'] >= -63.40) &
    (sucursales_carrefour['sucursales_longitud'] <= -56.60)
)

# CABA - AR-C
caba_ubi = (
    (sucursales_carrefour['sucursales_latitud'] >= -34.70) &
    (sucursales_carrefour['sucursales_latitud'] <= -34.55) &
    (sucursales_carrefour['sucursales_longitud'] >= -58.55) &
    (sucursales_carrefour['sucursales_longitud'] <= -58.35)
)

# Córdoba - AR-X
cordoba_ubi = (
    (sucursales_carrefour['sucursales_latitud'] >= -35.5) &
    (sucursales_carrefour['sucursales_latitud'] <= -29.0) &
    (sucursales_carrefour['sucursales_longitud'] >= -66.5) &
    (sucursales_carrefour['sucursales_longitud'] <= -61.0)
)

# Entre Ríos - AR-E
entre_rios_ubi = (
    (sucursales_carrefour['sucursales_latitud'] >= -34.0) &
    (sucursales_carrefour['sucursales_latitud'] <= -29.0) &
    (sucursales_carrefour['sucursales_longitud'] >= -61.5) &
    (sucursales_carrefour['sucursales_longitud'] <= -57.0)
)

# Santa Fe - AR-S
santa_fe_ubi = (
    (sucursales_carrefour['sucursales_latitud'] >= -34.5) &
    (sucursales_carrefour['sucursales_latitud'] <= -28.0) &
    (sucursales_carrefour['sucursales_longitud'] >= -62.5) &
    (sucursales_carrefour['sucursales_longitud'] <= -58.0)
)

# Corrientes - AR-W
corrientes_ubi = (
    (sucursales_carrefour['sucursales_latitud'] >= -30.5) &
    (sucursales_carrefour['sucursales_latitud'] <= -27.0) &
    (sucursales_carrefour['sucursales_longitud'] >= -59.5) &
    (sucursales_carrefour['sucursales_longitud'] <= -55.5)
)

# Salta - AR-A
salta_ubi = (
    (sucursales_carrefour['sucursales_latitud'] >= -26.5) &
    (sucursales_carrefour['sucursales_latitud'] <= -22.0) &
    (sucursales_carrefour['sucursales_longitud'] >= -66.5) &
    (sucursales_carrefour['sucursales_longitud'] <= -62.0)
)

# Jujuy - AR-Y
jujuy_ubi = (
    (sucursales_carrefour['sucursales_latitud'] >= -24.5) &
    (sucursales_carrefour['sucursales_latitud'] <= -21.5) &
    (sucursales_carrefour['sucursales_longitud'] >= -66.5) &
    (sucursales_carrefour['sucursales_longitud'] <= -64.0)
)


In [343]:
# Verificamos posibles inconsistencias geográficas entre la provincia declarada
# y la ubicación geográfica de las sucursales de Carrefour.

posibles_inconsistencias = (
    # Buenos Aires
    ((sucursales_carrefour['sucursales_provincia'] == 'AR-B') & ~bsas_ubi)  
    |
    ((sucursales_carrefour['sucursales_provincia'] == 'AR-C') & ~caba_ubi_2)  
    | 
    ((sucursales_carrefour['sucursales_provincia'] == 'AR-E') & ~entre_rios_ubi) 
    |
    ((sucursales_carrefour['sucursales_provincia'] == 'AR-S') & ~santa_fe_ubi)
    |
    ((sucursales_carrefour['sucursales_provincia'] == 'AR-W') & ~corrientes_ubi)
    |
    ((sucursales_carrefour['sucursales_provincia'] == 'AR-X') & ~cordoba_ubi)
    |
    ((sucursales_carrefour['sucursales_provincia'] == 'AR-A') & ~salta_ubi)
    |
    ((sucursales_carrefour['sucursales_provincia'] == 'AR-Y') & ~jujuy_ubi)
)

casos_revision = sucursales_carrefour.loc[
    posibles_inconsistencias,
    sucursales_carrefour.columns
].copy()


print(f"Casos detectados para revisión: {len(casos_revision)}")

display(casos_revision)

Casos detectados para revisión: 2


,id_comercio,id_bandera,id_sucursal,sucursales_nombre,sucursales_tipo,sucursales_calle,sucursales_numero,sucursales_latitud,sucursales_longitud,sucursales_observaciones,sucursales_barrio,sucursales_codigo_postal,sucursales_localidad,sucursales_provincia,sucursales_lunes_horario_atencion,sucursales_martes_horario_atencion,sucursales_miercoles_horario_atencion,sucursales_jueves_horario_atencion,sucursales_viernes_horario_atencion,sucursales_sabado_horario_atencion,sucursales_domingo_horario_atencion,provincia
67,10,3,172,San Juan 1128,Autoservicio,San Juan,1128,-32.890284,-68.837131,NaN,NaN,5500,Córdoba,AR-X,08:00 A 22:00,08:00 A 22:00,08:00 A 22:00,08:00 A 22:00,08:00 A 22:00,08:00 A 22:00,11:00 A 20:00,CORDOBA
664,10,3,854,Avenida Bartolomé Mitre 3125/3145,Autoservicio,Avenida Bartolomé Mitre,3125,-34.647420,58.794224,NaN,NaN,1744,Moreno,AR-B,08:00 A 22:00,08:00 A 22:00,08:00 A 22:00,08:00 A 22:00,08:00 A 22:00,08:00 A 22:00,11:00 A 20:00,BUENOS AIRES


La verificación geográfica arrojó dos casos con coordenadas fuera del rango esperado para su provincia declarada, que se analizaron por separado.

El primero corresponde a la sucursal `172`, cuya dirección y coordenadas confirman que pertenece a la provincia de **Mendoza** y no a la provincia previamente imputada. Se corrige la provincia correspondiente en el dataset de Carrefour.

El segundo corresponde a la sucursal `854` (Avenida Bartolomé Mitre 3125/3145, Moreno). En este caso la provincia declarada (`AR-B`, Buenos Aires) es correcta —Moreno es una localidad real de esa provincia—, pero su longitud está cargada con signo positivo (`58.79` en lugar de `-58.79`), lo que la ubica fuera del rango esperado y explica por qué el filtro geográfico la señaló. No se trata de un error de imputación territorial sino de un error de carga en la coordenada, aislado a este registro.

Con esto, la provincia queda confirmada para ambos casos: se corrige la de la sucursal `172`, y la de la sucursal `854` se mantiene sin cambios en uanto a la provincia pero vamos a corregir el error en la imputación de longitud. El conjunto de sucursales, según los filtros aplicados, queda geográficamente consistente para continuar con el análisis territorial.

#### 6.2 Corrección de inconsistencia geográfica.

In [344]:
# ID detectado en la revisión geográfica
id_sucursal_erronea = 172

# Corrección de la provincia declarada
sucursales_carrefour.loc[
    sucursales_carrefour['id_sucursal'] == id_sucursal_erronea,
    'sucursales_provincia'
] = 'AR-M'

# Corrección de la provincia normalizada
sucursales_carrefour.loc[
    sucursales_carrefour['id_sucursal'] == id_sucursal_erronea,
    'provincia'
] = 'MENDOZA'


In [345]:
sucursales_carrefour.loc[
    sucursales_carrefour['id_sucursal'] == '854', 'sucursales_longitud'
] = -58.794224

### 7. Identificación y exclusión de tiendas online

Para garantizar que el análisis geográfico refleje únicamente la presencia física de las cadenas, es necesario detectar y excluir las tiendas online.  
Las sucursales virtuales —como eCommerce, delivery o plataformas web— no poseen una ubicación territorial real y podrían distorsionar métricas regionales, densidades de sucursales y comparaciones entre provincias.

En este punto se realiza una búsqueda sistemática de registros que puedan corresponder a tiendas online tanto en Carrefour como en DIA, utilizando patrones textuales en los campos `sucursales_nombre` y `sucursales_tipo`.

#### 7.1 Búsqueda de tiendas online en Carrefour

In [346]:
# Búsqueda de registros potencialmente no correspondientes a sucursales físicas de venta.
patron_ecommerce = r'e[\s-]?commerce|online|web|delivery|virtual'

mask_nombre = (
    sucursales_carrefour['sucursales_nombre']
    .fillna('')
    .str.lower()
    .str.contains(patron_ecommerce, regex=True)
)

mask_tipo = (
    sucursales_carrefour['sucursales_tipo']
    .fillna('')
    .str.lower()
    .str.contains(patron_ecommerce, regex=True)
)

casos_carrefour = sucursales_carrefour[
    mask_nombre | mask_tipo
]

display(
    casos_carrefour[
        [
            'id_sucursal',
            'sucursales_nombre',
            'sucursales_tipo',
            'sucursales_provincia',
            'sucursales_localidad'
        ]
    ]
)

,id_sucursal,sucursales_nombre,sucursales_tipo,sucursales_provincia,sucursales_localidad


Se aplicaron patrones de búsqueda sobre los campos `sucursales_nombre` y `sucursales_tipo` para detectar posibles tiendas online.  
No se detectaron registros potencialmente asociados a canales online mediante los patrones textuales utilizados.  
Esto permite continuar el análisis territorial sin necesidad de realizar ajustes en esta cadena.

#### 7.2 Búsqueda de tiendas online en Dia.

In [347]:
mask_nombre = (
    sucursales_dia['sucursales_nombre']
    .fillna('')
    .str.lower()
    .str.contains(patron_ecommerce, regex=True)
)

mask_tipo = (
    sucursales_dia['sucursales_tipo']
    .fillna('')
    .str.lower()
    .str.contains(patron_ecommerce, regex=True)
)

casos_dia = sucursales_dia[
    mask_nombre | mask_tipo
]

display(
    casos_dia[
        [
            'id_sucursal',
            'sucursales_nombre',
            'sucursales_tipo',
            'sucursales_provincia',
            'sucursales_localidad'
        ]
    ]
)

,id_sucursal,sucursales_nombre,sucursales_tipo,sucursales_provincia,sucursales_localidad
982,90000,eCommerce,Supermercado,Buenos Aires,CABA


In [348]:
# ID detectado como tienda online
id_online_dia = 90000

# Eliminación del registro del dataset
sucursales_dia = sucursales_dia[
    sucursales_dia['id_sucursal'] != id_online_dia
].copy()


Se detectó un registro que, por su denominación, corresponde a eCommerce y se excluyó del análisis territorial.  

### 8. Confirmación del territorio común entre ambas cadenas

En esta etapa se verifica nuevamente el conjunto de provincias donde **DIA y Carrefour** tienen presencia simultánea.  
Si bien estas provincias ya habían sido identificadas previamente, es importante confirmar que —tras las correcciones y depuraciones realizadas en los datasets— la coincidencia territorial se mantiene intacta.

Esta verificación asegura que los filtros aplicados sobre las sucursales de ambas cadenas continúan alineados y que el análisis comparativo se realizará únicamente sobre provincias donde ambas operan de manera efectiva.

In [349]:
# Provincias presentes en cada dataset
nombre_provincias_dia = set(sucursales_dia['provincia'].unique())
nombre_provincias_carrefour = set(sucursales_carrefour['provincia'].unique())

# Intersección: provincias donde ambas cadenas tienen presencia
provincias_comunes = nombre_provincias_dia.intersection(nombre_provincias_carrefour)

provincias_comunes

{'BUENOS AIRES',
 'CABA',
 'CORDOBA',
 'CORRIENTES',
 'ENTRE RIOS',
 'JUJUY',
 'SALTA',
 'SANTA FE'}

In [350]:
# Cantidad de provincias por cadena
print(f"Cantidad de provincias en DIA: {len(nombre_provincias_dia)}")
print(f"Cantidad de provincias en Carrefour: {len(nombre_provincias_carrefour)}")
print(f"Provincias presentes en ambas cadenas: {len(provincias_comunes)}")
print(f"Provincias coincidentes: {sorted(provincias_comunes)}")


Cantidad de provincias en DIA: 9
Cantidad de provincias en Carrefour: 22
Provincias presentes en ambas cadenas: 8
Provincias coincidentes: ['BUENOS AIRES', 'CABA', 'CORDOBA', 'CORRIENTES', 'ENTRE RIOS', 'JUJUY', 'SALTA', 'SANTA FE']


In [351]:
# Creamos subconjuntos de sucursales de DIA y Carrefour que solo incluyan las provincias comunes a ambos datasets.
sucursales_dia_8 = sucursales_dia[
    sucursales_dia['provincia'].isin(provincias_comunes)
].copy()

sucursales_carrefour_8 = sucursales_carrefour[
    sucursales_carrefour['provincia'].isin(provincias_comunes)
].copy()

In [352]:
# Cantidad de sucursales por cadena
print("Sucursales Carrefour en provincias coincidentes:", sucursales_carrefour_8.shape[0])
print("Sucursales DIA en provincias coincidentes:", sucursales_dia_8.shape[0])

Sucursales Carrefour en provincias coincidentes: 648
Sucursales DIA en provincias coincidentes: 982


Con este filtrado, los datasets `sucursales_dia_8` y `sucursales_carrefour_8` quedan correctamente alineados y listos para el cruce posterior con el dataset de precios.


### 9. Integración de los datasets de productos con las sucursales en provincias coincidentes

En esta etapa se integran los datasets de productos con los datasets de sucursales previamente filtrados por provincias coincidentes.  
Antes de realizar el merge, se seleccionaron únicamente las columnas relevantes para el análisis, con el objetivo de depurar el esquema y evitar arrastrar información que no aporta al estudio territorial ni al análisis de precios.

La integración se realiza mediante un `inner merge` sobre la clave `id_sucursal`, lo que garantiza que:

- se conserven únicamente los productos pertenecientes a sucursales ubicadas en las provincias coincidentes,  
- se excluyan automáticamente las sucursales que no poseen productos dentro del subconjunto filtrado,  
- el dataset resultante quede estructurado y preparado para las etapas posteriores del análisis comparativo entre cadenas.

Este proceso permite consolidar la información de precios con la ubicación geográfica de cada sucursal, asegurando consistencia territorial y reduciendo el dataset a las variables estrictamente necesarias para el análisis.


In [353]:
# Seleccionamos las columnas relevantes para los datasets de sucursales y productos de ambas cadenas,
cols_sucursales = ['id_sucursal', 'provincia', 'sucursales_localidad', 'sucursales_latitud', 'sucursales_longitud', 'id_comercio']
cols_productos = ['id_sucursal','id_producto', 'productos_descripcion', 'productos_marca', 'precio_efectivo']
# Dia
sucursales_dia_8_sel = sucursales_dia_8[cols_sucursales].copy()
productos_dia_comunes_sel = productos_dia_comunes[cols_productos].copy()
# Carrefour
sucursales_carrefour_8_sel = sucursales_carrefour_8[cols_sucursales].copy()
productos_carrefour_comunes_sel = productos_carrefour_comunes[cols_productos].copy()


La selección de columnas responde a las necesidades de las etapas siguientes, no solo de esta:

- **`precio_efectivo`** es la variable central del análisis comparativo: como se definió en la Etapa 1, ya incorpora el mejor precio disponible entre el precio de lista y `promo1`, por lo que no es necesario conservar las columnas de precio originales.
- **`provincia`** y **`sucursales_localidad`** son las que sostienen todo el trabajo de esta etapa: la primera para agrupar y comparar, la segunda como referencia legible al inspeccionar resultados o detectar casos atípicos.
- **`sucursales_latitud`** y **`sucursales_longitud`** se conservan pensando en visualizaciones futuras (mapas de sucursales o de precios), aunque esta etapa no las utilice directamente.
- **`productos_descripcion`** y **`productos_marca`** no se usan en el análisis de precios por provincia, pero son el insumo de texto que va a necesitar la Etapa 4 para el etiquetado y la clasificación automática de rubros. Se conservan ahora para no tener que rehacer el cruce con sucursales más adelante.
- **`id_comercio`** se mantiene como referencia para poder distinguir de qué cadena proviene cada registro si en algún momento se combinan los datasets de DIA y Carrefour en una sola tabla.

In [354]:
# Integración de los datasets de productos con los datasets de sucursales.
# También validamos que la relación sea many-to-one: muchas filas de productos por cada sucursal única.
# En la la cadena Dia.
dia = productos_dia_comunes_sel.merge(
    sucursales_dia_8_sel,
    on='id_sucursal',
    how='inner',
    validate='many_to_one'
)


In [355]:
# Comprobamos que todas las sucursales del resultado de la integración estén en provincias coincidentes.
print("Sucursales únicas en el resultado:", dia['id_sucursal'].nunique())
print("¿Todas las sucursales están en provincias coincidentes?:",
      set(dia['provincia']).issubset(provincias_comunes))


Sucursales únicas en el resultado: 982
¿Todas las sucursales están en provincias coincidentes?: True


In [356]:
# Integración de los datasets de productos con los datasets de sucursales.
# También validamos que la relación sea many-to-one: muchas filas de productos por cada sucursal única.
# En la la cadena Carrefour.
carrefour = productos_carrefour_comunes_sel.merge(
    sucursales_carrefour_8_sel,
    on='id_sucursal',
    how='inner',
    validate='many_to_one'
)


In [357]:
print("Sucursales únicas en el resultado:", carrefour['id_sucursal'].nunique())
print("¿Todas las sucursales están en provincias coincidentes?:",
      set(carrefour['provincia']).issubset(provincias_comunes))

Sucursales únicas en el resultado: 633
¿Todas las sucursales están en provincias coincidentes?: True


In [358]:
sucursales_carrefour_sin_productos_comunes = (
    sucursales_carrefour_8[
        ~sucursales_carrefour_8['id_sucursal'].isin(
            productos_carrefour_comunes['id_sucursal']
        )
    ]
)

print("Sucursales de Carrefour sin productos comunes:", sucursales_carrefour_sin_productos_comunes.shape[0])
display(
    sucursales_carrefour_sin_productos_comunes[
        [
            'id_sucursal',
            'provincia',
            'sucursales_localidad',
            'sucursales_nombre'
        ]
    ]
)

Sucursales de Carrefour sin productos comunes: 15


,id_sucursal,provincia,sucursales_localidad,sucursales_nombre
49,581,CABA,Ciudad Autónoma de Buenos Aires,J. M. Gutiérrez 3759
172,536,CABA,Ciudad Autónoma de Buenos Aires,Ricardo Balbín 4189
280,112,SALTA,Salta,Salta
282,150,CABA,Ciudad Autónoma de Buenos Aires,Vidt
319,501,BUENOS AIRES,Virrey del Pino,MAXI - Virrey del Pino (501)
330,832,BUENOS AIRES,MAR DEL PLATA,MINI - LURO 3060 (832)
355,804,BUENOS AIRES,MAR DEL PLATA,ALBERTI 1511
457,735,CABA,Ciudad Autónoma de Buenos Aires,Pueyrredón 2120
471,801,BUENOS AIRES,MAR DEL PLATA,MINI - GUEMES 2470 (801)
514,461,BUENOS AIRES,Lomas de Zamora,Salta 160


### Resultados de la integración y verificación de consistencia

Una vez integrados los datasets de productos y sucursales, se evalúa la cantidad de sucursales presentes en cada resultado y se verifica que todas pertenezcan al territorio común definido en el punto anterior.

**Resultados obtenidos:**

- **DIA**  
  - Sucursales en provincias coincidentes: 982  
  - Sucursales presentes en el dataset integrado: 982  
  - ¿Todas las sucursales están en provincias coincidentes?: **True**

  Esto confirma que todas las sucursales de DIA dentro del territorio compartido poseen productos en el subconjunto utilizado. La integración conserva la totalidad del universo territorial filtrado.

- **Carrefour**  
  - Sucursales en provincias coincidentes: 648  
  - Sucursales presentes en el dataset integrado: 633  
  - ¿Todas las sucursales están en provincias coincidentes?: **True**

  En Carrefour, 633 sucursales tienen productos dentro del subconjunto filtrado (`productos_carrefour_comunes`), mientras que 15 sucursales no presentan registros de productos en dicho subconjunto. Esta reducción es esperable, dado que el filtrado previo conserva únicamente los productos coincidentes entre cadenas o los productos relevantes para este análisis.

### Aclaración metodológica sobre las 15 sucursales sin productos comunes

La verificación de estas 15 sucursales se realizó de manera puntual utilizando el dataset original completo (no incluido en este notebook para evitar recargas de archivos pesados).  
Al inspeccionar los productos asociados a dichas sucursales en el dataset original, se comprobó que presentan muy pocos registros de productos o directamente no poseen productos dentro del subconjunto filtrado utilizado en este análisis.

Por este motivo, su ausencia en el dataset integrado no representa un error del merge, sino una consecuencia natural del recorte analítico aplicado a los productos.  
Con esta validación, los datasets integrados (`dia` y `carrefour`) quedan consolidados y listos para avanzar hacia el análisis territorial y comparativo entre ambas cadenas.



### 9.1 Guardado de resultados intermedios

Las etapas siguientes parten de estos dos dataframes ya normalizados y cruzados. Se guardan en formato Parquet para no repetir la limpieza geográfica en cada notebook.

In [359]:
dia.to_parquet(
    '../data/processed/productos_dia_geo.parquet',
    index=False
)

carrefour.to_parquet(
    '../data/processed/productos_carrefour_geo.parquet',
    index=False
)
print("Resultados intermedios guardados en ../data/processed/")

Resultados intermedios guardados en ../data/processed/
